# Kordiam Report parser – Parser GOLIVE v12

Dit notebook verwerkt een **Kordiam Report** (.xlsx) naar **Verhalenaanbod** volgens de pipeline.


In [ ]:
from __future__ import annotations

import os
import datetime
from pathlib import Path
from typing import Dict, List, Tuple

import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side


In [ ]:
# === Upload bestanden (.xlsx) ===
# 1) Upload je Kordiam Report bestand.
# 2) Mappingregels parser.xlsx is een VAST bestand. Je kunt:
#    - het éénmalig uploaden (optioneel), of
#    - het lokaal in ./assets/Mappingregels parser.xlsx zetten.
# 3) Na upload worden INPUT_XLSX / OUTPUT_XLSX / MAPPING_XLSX automatisch gezet.
# 4) Run daarna de volgende cellen.

from IPython.display import display
import ipywidgets as widgets

UPLOAD_DIR = Path("./uploads")
ASSET_DIR = Path("./assets")
UPLOAD_DIR.mkdir(exist_ok=True)
ASSET_DIR.mkdir(exist_ok=True)

upload_kordiam = widgets.FileUpload(accept=".xlsx", multiple=False, description="Upload Kordiam Report")
upload_mapping = widgets.FileUpload(accept=".xlsx", multiple=False, description="Upload mappingregels (optioneel)")
output_name = widgets.Text(value="Verhalenaanbod.xlsx", description="Output naam:")
status = widgets.HTML(value="<i>Upload nog geen bestand.</i>")

def _get_upload_payload(upl):
    if not upl.value:
        return None, None
    if isinstance(upl.value, dict):  # ipywidgets v8
        filename = next(iter(upl.value.keys()))
        content = upl.value[filename]["content"]
        return filename, content
    # ipywidgets v7
    filename = upl.value[0]["name"]
    content = upl.value[0]["content"]
    return filename, content

def _save_uploads(_change=None):
    global INPUT_XLSX, OUTPUT_XLSX, MAPPING_XLSX

    # Kordiam (verplicht)
    k_name, k_content = _get_upload_payload(upload_kordiam)
    if k_name and k_content:
        safe_name = k_name.replace("/", "_").replace("\\", "_")
        input_path = UPLOAD_DIR / safe_name
        input_path.write_bytes(k_content)
        INPUT_XLSX = str(input_path.resolve())

    # Mappingregels (optioneel / vast asset)
    m_name, m_content = _get_upload_payload(upload_mapping)
    if m_name and m_content:
        safe_name = m_name.replace("/", "_").replace("\\", "_")
        map_path = ASSET_DIR / safe_name
        map_path.write_bytes(m_content)
        MAPPING_XLSX = str(map_path.resolve())
    else:
        default_map = (ASSET_DIR / "Mappingregels parser.xlsx").resolve()
        if "MAPPING_XLSX" not in globals():
            MAPPING_XLSX = str(default_map)

    OUTPUT_XLSX = str(Path(output_name.value).resolve())

    status.value = (
        f"<b>✅ Status</b><br>"
        f"INPUT_XLSX = <code>{globals().get('INPUT_XLSX','')}</code><br>"
        f"MAPPING_XLSX = <code>{globals().get('MAPPING_XLSX','')}</code><br>"
        f"OUTPUT_XLSX = <code>{globals().get('OUTPUT_XLSX','')}</code>"
    )

upload_kordiam.observe(_save_uploads, names="value")
upload_mapping.observe(_save_uploads, names="value")
output_name.observe(_save_uploads, names="value")

display(widgets.VBox([upload_kordiam, upload_mapping, output_name, status]))


In [ ]:
# === Instellingen (fallback) ===
# Als je liever geen upload-widget gebruikt, kun je hier handmatig paden invullen.
# (Let op: als je de uploadcel hierboven gebruikt, overschrijft die INPUT_XLSX/OUTPUT_XLSX/MAPPING_XLSX automatisch.)

INPUT_XLSX = globals().get("INPUT_XLSX", "")
OUTPUT_XLSX = globals().get("OUTPUT_XLSX", "Verhalenaanbod.xlsx")

# Vast asset-bestand (niet elke keer uploaden). Standaard verwacht in ./assets/
MAPPING_XLSX = globals().get("MAPPING_XLSX", str((Path("./assets") / "Mappingregels parser.xlsx").resolve()))


In [ ]:
TARGET_SHEET = "Totale verhalenlijst"
SOURCE_SHEET = "Story List"

SHEETS_TO_REMOVE = ["Statistics", "Aggregated story list"]

# Headers in A1.. (dynamisch op basis van deze lijst)
# NB: 'Classificatie' hoort tussen Focusregio en Heel Limburg.
TARGET_HEADERS: List[str] = ["Story ID", "Naam productie", "Note", "Publ. status", "Focusregio", "Classificatie", "Heel Limburg", "Voorkeurspositie", "Beeld voor print", "Publicatiedwang", "Top 8", "Karakters", "Artikelsoort", "Leverancier", "Auteur", "Gewenste placeholder", "Tweede keus placeholder", "Derde keus placeholder", "Vierde keus placeholder", "Placeholder bij enigszins geschikt", "Prioscore", "Gekozen template", "Gekozen placeholder", "Placeholder-concessie", "Plaatsing"]

# Mapping: doelkolom -> bronkolom (op headernaam). Niet-gemapte kolommen blijven leeg.
COL_MAP: Dict[str, str] = {
    "Story ID": "Story ID",
    "Naam productie": "Description",
    "Note": "Note",
    "Publ. status": "Publ. status",
    "Focusregio": "Focusregio",
    "Classificatie": "Classificatie",
    "Heel Limburg": "Heel Limburg",
    "Voorkeurspositie": "Voorkeurspositie",
    "Beeld voor print": "Beeld voor print",
    "Publicatiedwang": "Publicatiedwang",
    "Top 8": "Top 8",
    "Karakters": "Text length"
}

# Fixed text in AA1 en AB1 (geen headers)
FIXED_TEXT = "Totale verhalenlijst"


In [ ]:
def find_header_row(ws, needle: str = "Story ID", max_scan_rows: int = 200) -> int:
    for r in range(1, max_scan_rows + 1):
        row_vals = [c.value for c in ws[r]]
        if needle in row_vals:
            return r
    raise ValueError(f"Kon header-rij niet vinden: '{needle}' niet gevonden in eerste {max_scan_rows} rijen.")


def build_header_index(ws, header_row: int) -> Dict[str, int]:
    idx: Dict[str, int] = {}
    for col_i, cell in enumerate(ws[header_row], start=1):
        val = cell.value
        if isinstance(val, str) and val.strip():
            idx[val.strip()] = col_i
    return idx


def clear_or_create_sheet(wb: openpyxl.Workbook, name: str):
    if name in wb.sheetnames:
        wb.remove(wb[name])
    return wb.create_sheet(title=name)


def write_target_headers(ws_target, headers: List[str]) -> None:
    # Opmaak: Bold + 20% grijs
    header_font = Font(bold=True)
    header_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    for i, h in enumerate(headers, start=1):
        cell = ws_target.cell(row=1, column=i, value=h)
        cell.font = header_font
        cell.fill = header_fill


def set_fixed_text(ws_target) -> None:
    ws_target.cell(row=1, column=27, value=FIXED_TEXT)  # AA1
    ws_target.cell(row=1, column=28, value=FIXED_TEXT)  # AB1


def iter_data_rows(ws_source, header_row: int):
    r = header_row + 1
    while True:
        row = ws_source[r]
        values = [c.value for c in row]
        if all(v is None or (isinstance(v, str) and not v.strip()) for v in values):
            break
        yield r
        r += 1


def process_kordiam(input_xlsx: str, output_xlsx: str, mapping_xlsx: str = "") -> Tuple[int, List[str]]:
    if not os.path.exists(input_xlsx):
        raise FileNotFoundError(f"Inputbestand niet gevonden: {input_xlsx}")

    wb = openpyxl.load_workbook(input_xlsx)

    # v26: parse planning date from 'Story List'!A3 (last 10 chars)
    planning_date = None
    try:
        if SOURCE_SHEET in wb.sheetnames:
            raw_a3 = wb[SOURCE_SHEET]["A3"].value
            if raw_a3 is not None:
                s = str(raw_a3)
                s10 = s[-10:].replace(".", "-")
                for fmt in ("%d-%m-%Y", "%Y-%m-%d"):
                    try:
                        planning_date = datetime.datetime.strptime(s10, fmt).date()
                        break
                    except Exception:
                        pass
    except Exception:
        planning_date = None


    # 1) remove sheets
    for s in SHEETS_TO_REMOVE:
        if s in wb.sheetnames:
            wb.remove(wb[s])

    if SOURCE_SHEET not in wb.sheetnames:
        raise ValueError(f"Bron-sheet '{SOURCE_SHEET}' ontbreekt. Aanwezige sheets: {wb.sheetnames}")
    ws_source = wb[SOURCE_SHEET]

    # 2) create target sheet
    ws_target = clear_or_create_sheet(wb, TARGET_SHEET)
    write_target_headers(ws_target, TARGET_HEADERS)

    # 3) AA1/AB1 fixed text
    set_fixed_text(ws_target)

    header_row = find_header_row(ws_source, needle="Story ID")
    src_idx = build_header_index(ws_source, header_row)

    warnings: List[str] = []
    if mapping_xlsx and not os.path.exists(mapping_xlsx):
        warnings.append(f"WAARSCHUWING: mappingbestand niet gevonden: {mapping_xlsx} (stap 9 wordt overgeslagen)")
    elif not mapping_xlsx:
        warnings.append("WAARSCHUWING: mappingbestand (MAPPING_XLSX) is leeg/niet gezet (stap 9 wordt overgeslagen)")

    required_sources = set(COL_MAP.values()) | {"Assignee last name", "Assignee first name", "Type verhaal", "Group"}
    missing = sorted({src for src in required_sources if src not in src_idx})
    for m in missing:
        warnings.append(f"WAARSCHUWING: bronkolom ontbreekt in '{SOURCE_SHEET}': {m} (kolom wordt leeg gelaten)")

    # 9) Placeholder-mapping voorbereiden
    placeholder_lookup = {}
    if mapping_xlsx and os.path.exists(mapping_xlsx):
        try:
            wb_map = openpyxl.load_workbook(mapping_xlsx, data_only=True)
            ws_map = None
            for name in wb_map.sheetnames:
                ws_try = wb_map[name]
                header = [ws_try.cell(1, c).value for c in range(1, ws_try.max_column + 1)]
                header_norm = [h.strip() if isinstance(h, str) else h for h in header]
                if "Artikelsoort" in header_norm and "Beeld voor print" in header_norm and "Top 8" in header_norm:
                    ws_map = ws_try
                    break
            if ws_map is None:
                warnings.append(f"WAARSCHUWING: mappingbestand heeft geen herkenbare header (stap 9 wordt overgeslagen): {mapping_xlsx}")
            else:
                header = [ws_map.cell(1, c).value for c in range(1, ws_map.max_column + 1)]
                header_norm = [h.strip() if isinstance(h, str) else h for h in header]
                col_idx = {h: i + 1 for i, h in enumerate(header_norm) if isinstance(h, str) and h}

                def _norm(v):
                    if v is None:
                        return "(geen waarde ingevuld)"
                    if isinstance(v, str):
                        s = v.strip()
                        return s if s else "(geen waarde ingevuld)"
                    return str(v).strip()

                for rr in range(2, ws_map.max_row + 1):
                    artikel = ws_map.cell(rr, col_idx.get("Artikelsoort", 1)).value
                    if artikel is None or (isinstance(artikel, str) and not artikel.strip()):
                        break
                    beeld = ws_map.cell(rr, col_idx.get("Beeld voor print", 2)).value
                    top8 = ws_map.cell(rr, col_idx.get("Top 8", 3)).value
                    key = (_norm(artikel), _norm(beeld), _norm(top8))
                    placeholder_lookup[key] = {
                        "Gewenste placeholder": ws_map.cell(rr, col_idx.get("Gewenste placeholder", 0)).value if col_idx.get("Gewenste placeholder") else None,
                        "Tweede keus placeholder": ws_map.cell(rr, col_idx.get("Tweede keus placeholder", 0)).value if col_idx.get("Tweede keus placeholder") else None,
                        "Derde keus placeholder": ws_map.cell(rr, col_idx.get("Derde keus placeholder", 0)).value if col_idx.get("Derde keus placeholder") else None,
                        "Vierde keus placeholder": ws_map.cell(rr, col_idx.get("Vierde keus placeholder", 0)).value if col_idx.get("Vierde keus placeholder") else None,
                        "Placeholder bij enigszins geschikt": ws_map.cell(rr, col_idx.get("Placeholder bij enigszins geschikt", 0)).value if col_idx.get("Placeholder bij enigszins geschikt") else None,
                    }
        except Exception as e:
            warnings.append(f"WAARSCHUWING: kon mappingbestand niet lezen ({mapping_xlsx}); stap 9 wordt overgeslagen. Fout: {e}")

    unmapped_placeholder_rows = 0

    out_row = 2
    mapped_rows = 0
    removed_rows = 0

    for r in iter_data_rows(ws_source, header_row):
        story_id_col = src_idx.get("Story ID")
        story_id_val = ws_source.cell(row=r, column=story_id_col).value if story_id_col else None
        if story_id_val is None or (isinstance(story_id_val, str) and not story_id_val.strip()):
            break

        # 10) filter Column/Rubriek
        col_tv_filter = src_idx.get("Type verhaal")
        tv_filter_val = ws_source.cell(row=r, column=col_tv_filter).value if col_tv_filter is not None else None
        tv_filter_str = tv_filter_val.strip() if isinstance(tv_filter_val, str) else (str(tv_filter_val).strip() if tv_filter_val is not None else "")
        if tv_filter_str in {"Column", "Rubriek"}:
            removed_rows += 1
            continue

        # basis mapping
        for target_col_i, target_header in enumerate(TARGET_HEADERS, start=1):
            src_header = COL_MAP.get(target_header)
            if src_header is None:
                continue
            src_col_i = src_idx.get(src_header)
            if src_col_i is None:
                continue
            ws_target.cell(row=out_row, column=target_col_i, value=ws_source.cell(row=r, column=src_col_i).value)

        # 6) Auteur
        auteur_last = ws_source.cell(row=r, column=src_idx["Assignee last name"]).value if "Assignee last name" in src_idx else None
        auteur_first = ws_source.cell(row=r, column=src_idx["Assignee first name"]).value if "Assignee first name" in src_idx else None
        auteur_last = auteur_last.strip() if isinstance(auteur_last, str) else auteur_last
        auteur_first = auteur_first.strip() if isinstance(auteur_first, str) else auteur_first
        if auteur_last and auteur_first:
            auteur_val = f"{auteur_last}, {auteur_first}"
        elif auteur_last:
            auteur_val = str(auteur_last)
        elif auteur_first:
            auteur_val = str(auteur_first)
        else:
            auteur_val = None
        ws_target.cell(row=out_row, column=TARGET_HEADERS.index("Auteur") + 1, value=auteur_val)

        # 7) Artikelsoort
        text_len_val = ws_source.cell(row=r, column=src_idx["Text length"]).value if "Text length" in src_idx else None
        type_verhaal_val = ws_source.cell(row=r, column=src_idx["Type verhaal"]).value if "Type verhaal" in src_idx else None
        try:
            tl_int = int(float(str(text_len_val).strip())) if text_len_val is not None and str(text_len_val).strip() != "" else None
        except Exception:
            tl_int = None
        tv_str = str(type_verhaal_val).strip() if type_verhaal_val is not None else ""
        artikelsoort = None
        if tl_int == 7200:
            artikelsoort = "XXL"
        elif tl_int == 5400:
            artikelsoort = "XL"
        elif tl_int == 4000:
            artikelsoort = "L"
        elif tl_int == 2800:
            artikelsoort = "M_nws" if tv_str == "Nieuws" else "M_lk"
        elif tl_int == 1800:
            artikelsoort = "S_nws" if tv_str == "Nieuws" else "S_lk"
        elif tl_int == 1000:
            artikelsoort = "XS"
        ws_target.cell(row=out_row, column=TARGET_HEADERS.index("Artikelsoort") + 1, value=artikelsoort)

        # 8) Leverancier
        group_val = ws_source.cell(row=r, column=src_idx["Group"]).value if "Group" in src_idx else None
        supplier_map = {
            "Nieuwsdienst": "rND",
            "Maastricht - Heuvelland": "rMH",
            "Sittard-Geleen": "rSG",
            "Parkstad": "rPS",
            "Noord-Limburg": "rNO",
            "Midden-Limburg": "rMI",
            "Economie": "rEC",
            "Cultuur & Media": "rCU",
            "LS": "rLS",
            "Onderzoek": "rOZ",
            "Opinie": "rOP",
            "Sport": "rSP",
        }
        leverancier_val = supplier_map.get(str(group_val).strip() if group_val is not None else None)
        ws_target.cell(row=out_row, column=TARGET_HEADERS.index("Leverancier") + 1, value=leverancier_val)

        # 9) Placeholders
        if placeholder_lookup:
            def _norm(v):
                if v is None:
                    return "(geen waarde ingevuld)"
                if isinstance(v, str):
                    s = v.strip()
                    return s if s else "(geen waarde ingevuld)"
                return str(v).strip()

            beeld_val = ws_source.cell(row=r, column=src_idx["Beeld voor print"]).value if "Beeld voor print" in src_idx else None
            top8_val = ws_source.cell(row=r, column=src_idx["Top 8"]).value if "Top 8" in src_idx else None
            key = (_norm(artikelsoort), _norm(beeld_val), _norm(top8_val))
            rule = placeholder_lookup.get(key)
            if rule:
                for h, v in rule.items():
                    if h in TARGET_HEADERS:
                        ws_target.cell(row=out_row, column=TARGET_HEADERS.index(h) + 1, value=v)
            else:
                unmapped_placeholder_rows += 1

        mapped_rows += 1
        out_row += 1

    if placeholder_lookup and unmapped_placeholder_rows:
        warnings.append(f"WAARSCHUWING: stap 9 (placeholder mapping) geen match gevonden voor {unmapped_placeholder_rows} rij(en); placeholders zijn daar leeg gelaten.")
    if removed_rows:
        warnings.append(f"INFO: stap 10 (filter Column/Rubriek) verwijderd: {removed_rows} rij(en).")

    # 11) verwijder Story List
    if SOURCE_SHEET in wb.sheetnames:
        wb.remove(wb[SOURCE_SHEET])

    # Kandidatenlijsten maken
    candidate_names = ["GO-01", "GO-02", "NM-NO", "NM-MI", "ZU-SG", "ZU-PS", "ZU-MH"]
    for nm in candidate_names:
        if nm in wb.sheetnames:
            wb.remove(wb[nm])
    base_ws = wb[TARGET_SHEET]
    for nm in candidate_names:
        ws_copy = wb.copy_worksheet(base_ws)
        ws_copy.title = nm

    # Kandidatenlijsten maken: opmaak + fixed texts
    gray50 = PatternFill(start_color="808080", end_color="808080", fill_type="solid")
    thick_side = Side(style="thick")
    fixed_by_sheet = {
        "GO-01": ("rND; rOZ; rEC;", "Limburg-breed"),
        "GO-02": ("rND; rOZ; rEC;", "Limburg-breed"),
        "NM-NO": ("rNO", "Noord"),
        "NM-MI": ("rMI", "Midden"),
        "ZU-SG": ("rSG", "Sittard"),
        "ZU-PS": ("rPS", "Parkstad"),
        "ZU-MH": ("rMH", "Maastricht"),
    }

    def _apply_right_border(cell, side):
        b = cell.border
        cell.border = Border(
            left=b.left, right=side, top=b.top, bottom=b.bottom,
            diagonal=b.diagonal, diagonal_direction=b.diagonal_direction,
            outline=b.outline, vertical=b.vertical, horizontal=b.horizontal
        )

    for nm in candidate_names:
        ws_c = wb[nm]

        # Verticale lijn tussen Z en AA: dikke rechterborder op kolom Z
        max_r = max(ws_c.max_row, 40)
        for rr in range(1, max_r + 1):
            _apply_right_border(ws_c.cell(row=rr, column=26), thick_side)

        # AA1:AF40 50% grijs
        for rr in range(1, 41):
            for cc in range(27, 33):  # AA..AF
                ws_c.cell(row=rr, column=cc).fill = gray50

        # Fixed text AE1/AF1
        if nm in fixed_by_sheet:
            ae1, af1 = fixed_by_sheet[nm]
            ws_c.cell(row=1, column=31, value=ae1)  # AE1
            ws_c.cell(row=1, column=32, value=af1)  # AF1

        # Kandidatenlijsten voor regiospreads bewerken (NM-NO, NM-MI, ZU-SG, ZU-PS, ZU-MH)
        if nm in {"NM-NO", "NM-MI", "ZU-SG", "ZU-PS", "ZU-MH"}:
            focus_col = TARGET_HEADERS.index("Focusregio") + 1
            af1_val = ws_c.cell(row=1, column=32).value  # AF1
            af1_key = str(af1_val).strip().lower() if af1_val is not None else ""
            keep_alt = "limburg-breed"

            # 1) verwijderen: focus bevat niet ab2 én niet limburg-breed
            for rr in range(ws_c.max_row, 1, -1):
                focus_val = ws_c.cell(row=rr, column=focus_col).value
                focus_str = str(focus_val).strip().lower() if focus_val is not None else ""
                if (af1_key not in focus_str) and (keep_alt not in focus_str):
                    ws_c.delete_rows(rr, 1)

            # 2) verwijderen: focus bevat limburg-breed + extra waarde, maar niet ab2
            for rr in range(ws_c.max_row, 1, -1):
                focus_val = ws_c.cell(row=rr, column=focus_col).value
                focus_str = str(focus_val).strip().lower() if focus_val is not None else ""
                if keep_alt in focus_str and (af1_key not in focus_str):
                    norm = focus_str
                    for sep in [",", "\n", "/", "|"]:
                        norm = norm.replace(sep, ";")
                    parts = [p.strip() for p in norm.split(";") if p and p.strip()]
                    if ("limburg-breed" in parts) and (len(set(parts)) >= 2):
                        ws_c.delete_rows(rr, 1)

            # 3) classificatie instellen
            class_col = TARGET_HEADERS.index("Classificatie") + 1
            chars_col = TARGET_HEADERS.index("Karakters") + 1
            for rr in range(2, ws_c.max_row + 1):
                focus_val = ws_c.cell(row=rr, column=focus_col).value
                focus_str = str(focus_val).strip().lower() if focus_val is not None else ""
                chars_val = ws_c.cell(row=rr, column=chars_col).value
                try:
                    chars_num = int(float(str(chars_val).strip())) if chars_val is not None and str(chars_val).strip() != "" else None
                except Exception:
                    chars_num = None

                new_class = None
                if af1_key and (af1_key in focus_str):
                    new_class = "A-keus; B-keus; C-keus;"
                else:
                    if focus_str == "limburg-breed":
                        if chars_num is not None and chars_num > 4001:
                            new_class = "C-keus"
                        else:
                            new_class = "B-keus; C-keus"

                if new_class is not None:
                    ws_c.cell(row=rr, column=class_col, value=new_class)
            # 4) Prioscore berekenen (beginwaarde 0)
            prio_col = TARGET_HEADERS.index("Prioscore") + 1
            top8_col = TARGET_HEADERS.index("Top 8") + 1
            pubd_col = TARGET_HEADERS.index("Publicatiedwang") + 1
            hl_col = TARGET_HEADERS.index("Heel Limburg") + 1
            pref_col = TARGET_HEADERS.index("Voorkeurspositie") + 1
            lev_col = TARGET_HEADERS.index("Leverancier") + 1
            ae1_val = ws_c.cell(row=1, column=31).value  # AE1
            ae1_str = str(ae1_val).strip().lower() if ae1_val is not None else ""

            for rr in range(2, ws_c.max_row + 1):
                score = 0

                class_val = ws_c.cell(row=rr, column=class_col).value
                class_str = str(class_val).lower() if class_val is not None else ""

                top8_val = ws_c.cell(row=rr, column=top8_col).value
                top8_str = str(top8_val).strip().lower() if top8_val is not None else ""

                pubd_val = ws_c.cell(row=rr, column=pubd_col).value
                pubd_str = str(pubd_val).strip().lower() if pubd_val is not None else ""

                hl_val = ws_c.cell(row=rr, column=hl_col).value
                hl_str = str(hl_val).strip().lower() if hl_val is not None else ""

                pref_val = ws_c.cell(row=rr, column=pref_col).value

                if top8_str == "ja" and ("a-keus" in class_str):
                    score += 2
                if "a-keus" in class_str:
                    score += 5

                if hl_str == "moet mee" and ("a-keus" in class_str):
                    score += 3

                if pubd_str == "ja":
                    score += 3
                elif pubd_str == "nee":
                    score -= 1
                elif pubd_str == "op te sparen":
                    score += 1

                # Leverancier-regel: IF Leverancier = waarde cel AE1 THEN +1
                lev_val = ws_c.cell(row=rr, column=lev_col).value
                lev_str = str(lev_val).strip().lower() if lev_val is not None else ""
                if ae1_str and lev_str == ae1_str:
                    score += 1

                # Voorkeurspositie-regels t.o.v. naam tabblad (sheet)
                sheet_name = nm
                if pref_val is not None and str(pref_val).strip() != "":
                    pref_str = str(pref_val).strip()
                    if pref_str == sheet_name:
                        score += 20
                    else:
                        # v13 fix: alleen -20 als Voorkeurspositie niet 'Nee' is en niet de sheetnaam
                        if pref_str.lower() != "nee":
                            score -= 20

                ws_c.cell(row=rr, column=prio_col, value=score)

            # 5) Sorteer op Prioscore (hoog -> laag) binnen het tabblad (alleen data A..Y; fixed blok AA..AF blijft staan)
            data_cols = len(TARGET_HEADERS)  # A..Y
            rows_data = []
            for rr in range(2, ws_c.max_row + 1):
                row_vals = [ws_c.cell(row=rr, column=cc).value for cc in range(1, data_cols + 1)]
                prio_val = row_vals[prio_col - 1]
                try:
                    prio_num = float(prio_val)
                except Exception:
                    prio_num = float("-inf")
                rows_data.append((prio_num, row_vals))

            rows_data.sort(key=lambda x: x[0], reverse=True)

            for i, (_prio, row_vals) in enumerate(rows_data, start=2):
                for cc, v in enumerate(row_vals, start=1):
                    c = ws_c.cell(row=i, column=cc)
                    c.value = v


            # Re-apply grijs blok + fixed texts
            for rr in range(1, 41):
                for cc in range(27, 33):
                    ws_c.cell(row=rr, column=cc).fill = gray50
            # (v15+) AA2/AB2 worden niet meer gebruikt; leeg laten.
            ws_c.cell(row=2, column=27, value=None)
            ws_c.cell(row=2, column=28, value=None)



        # Kandidatenlijsten voor GO-spreads bewerken (GO-01, GO-02) (v17a)
        if nm in {"GO-01", "GO-02"}:
            heel_col = TARGET_HEADERS.index("Heel Limburg") + 1
            focus_col = TARGET_HEADERS.index("Focusregio") + 1
            kar_col = TARGET_HEADERS.index("Karakters") + 1
            class_col = TARGET_HEADERS.index("Classificatie") + 1

            # 1) Verwijder alle rijen waarvoor geldt: Heel Limburg≠geschikt AND Heel Limburg≠moet mee
            for rr in range(ws_c.max_row, 1, -1):
                heel_val = ws_c.cell(row=rr, column=heel_col).value
                heel_key = str(heel_val).strip().lower() if heel_val is not None else ""
                if heel_key not in {"geschikt", "moet mee"}:
                    ws_c.delete_rows(rr, 1)

            # 2) Bepaal per rij de waarde voor Classificatie (GO-regels)
            leverancier_col = TARGET_HEADERS.index("Leverancier") + 1
            for rr in range(2, ws_c.max_row + 1):
                lev_val = ws_c.cell(row=rr, column=leverancier_col).value
                lev_key = str(lev_val).strip().lower() if lev_val is not None else ""

                focus_val = ws_c.cell(row=rr, column=focus_col).value
                focus_key = str(focus_val).strip().lower() if focus_val is not None else ""

                kar_val = ws_c.cell(row=rr, column=kar_col).value
                try:
                    kar_num = int(float(kar_val)) if kar_val is not None and str(kar_val).strip() != "" else None
                except Exception:
                    kar_num = None

                # v18:
                # IF Leverancier=rND OR rOZ OR rEC THEN A/B/C
                # IF Focusregio CONTAINS 'Limburg-breed' THEN A/B/C
                # ELSE B/C; and if Karakters>4001 THEN C
                if lev_key in {"rnd", "roz", "rec"}:
                    class_val = "A-keus; B-keus; C-keus;"
                elif "limburg-breed" in focus_key:
                    class_val = "A-keus; B-keus; C-keus;"
                else:
                    class_val = "B-keus; C-keus;"
                    if kar_num is not None and kar_num > 4001:
                        class_val = "C-keus;"

                ws_c.cell(row=rr, column=class_col, value=class_val)

            # 4) Prioscore berekenen (beginwaarde 0) (GO-01/GO-02 v19)
            prio_col = TARGET_HEADERS.index("Prioscore") + 1
            top8_col = TARGET_HEADERS.index("Top 8") + 1
            pubd_col = TARGET_HEADERS.index("Publicatiedwang") + 1
            hl_col = TARGET_HEADERS.index("Heel Limburg") + 1
            pref_col = TARGET_HEADERS.index("Voorkeurspositie") + 1
            lev_col = TARGET_HEADERS.index("Leverancier") + 1

            for rr in range(2, ws_c.max_row + 1):
                score = 0

                class_val = ws_c.cell(row=rr, column=class_col).value
                class_str = str(class_val).lower() if class_val is not None else ""

                top8_val = ws_c.cell(row=rr, column=top8_col).value
                top8_str = str(top8_val).strip().lower() if top8_val is not None else ""

                pubd_val = ws_c.cell(row=rr, column=pubd_col).value
                pubd_str = str(pubd_val).strip().lower() if pubd_val is not None else ""

                hl_val = ws_c.cell(row=rr, column=hl_col).value
                hl_str = str(hl_val).strip().lower() if hl_val is not None else ""

                pref_val = ws_c.cell(row=rr, column=pref_col).value

                if top8_str == "ja" and ("a-keus" in class_str):
                    score += 2
                if "a-keus" in class_str:
                    score += 5

                if hl_str == "moet mee" and ("a-keus" in class_str):
                    score += 3

                if pubd_str == "ja":
                    score += 3
                elif pubd_str == "nee":
                    score -= 1
                elif pubd_str == "op te sparen":
                    score += 1

                # Leverancier-regel GO: IF Leverancier=rND OR rOZ OR rEC THEN +1
                lev_val = ws_c.cell(row=rr, column=lev_col).value
                lev_str = str(lev_val).strip().lower() if lev_val is not None else ""
                if lev_str in {"rnd", "roz", "rec"}:
                    score += 1

                # Voorkeurspositie-regels t.o.v. naam tabblad (sheet)
                sheet_name = nm
                if pref_val is not None and str(pref_val).strip() != "":
                    pref_str = str(pref_val).strip()
                    if pref_str == sheet_name:
                        score += 20
                    else:
                        if pref_str.lower() != "nee":
                            score -= 20

                ws_c.cell(row=rr, column=prio_col, value=score)

            # 5) Sorteer op Prioscore (hoog -> laag) binnen het tabblad (alleen data A..Y; fixed blok AA..AF blijft staan)
            data_cols = len(TARGET_HEADERS)  # A..Y
            rows_data = []
            for rr in range(2, ws_c.max_row + 1):
                row_vals = [ws_c.cell(row=rr, column=cc).value for cc in range(1, data_cols + 1)]
                prio_val = row_vals[prio_col - 1]
                try:
                    prio_num = float(prio_val)
                except Exception:
                    prio_num = float("-inf")
                rows_data.append((prio_num, row_vals))

            rows_data.sort(key=lambda x: x[0], reverse=True)

            for i, (_prio, row_vals) in enumerate(rows_data, start=2):
                for cc, v in enumerate(row_vals, start=1):
                    c = ws_c.cell(row=i, column=cc)
                    c.value = v


    # === Tabblad Stats maken en voorbereiden (v16) ===
    stats_name = "Stats"
    if stats_name in wb.sheetnames:
        wb.remove(wb[stats_name])
    ws_stats = wb.create_sheet(title=stats_name)

    # v26: planning date in Stats!AA1
    if planning_date is not None:
        cdt = ws_stats["AA1"]
        cdt.value = planning_date
        cdt.number_format = "yyyy-mm-dd"

    stats_headers = ["Tabblad", "St_aantal", "St_karakters", "St_fotos_dr", "St_fotos_overig", "St_fotos_flex", "Complexiteit", "Akpp_ondergrens", "Akpp_range", "Akpp_range_boost"]
    header_font = Font(bold=True)
    header_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    for ci, h in enumerate(stats_headers, start=1):
        cell = ws_stats.cell(row=1, column=ci, value=h)
        cell.font = header_font
        cell.fill = header_fill

    ordered_tabs = ["GO-01", "GO-02", "NM-NO", "NM-MI", "ZU-SG", "ZU-PS", "ZU-MH"]
    ordered_tabs = [t for t in ordered_tabs if t in wb.sheetnames]

    complexity_map = {}
    akpp_range_map = {}
    akpp_range_boost_map = {}

    def _get_col(ws, header_name: str):
        max_c = ws.max_column
        headers = [ws.cell(1, c).value for c in range(1, max_c + 1)]
        try:
            return headers.index(header_name) + 1
        except ValueError:
            return None

    def _count_a_keus(ws):
        class_col = _get_col(ws, "Classificatie")
        if not class_col:
            return 0
        cnt = 0
        target = "A-keus; B-keus; C-keus;"
        for rr in range(2, ws.max_row + 1):
            v = ws.cell(rr, class_col).value
            if v == target:
                cnt += 1
        return cnt

    def _sum_karakters_a_keus(ws):
        class_col = _get_col(ws, "Classificatie")
        kar_col = _get_col(ws, "Karakters")
        if not class_col or not kar_col:
            return 0
        target = "A-keus; B-keus; C-keus;"
        total = 0
        for rr in range(2, ws.max_row + 1):
            if ws.cell(rr, class_col).value == target:
                kv = ws.cell(rr, kar_col).value
                try:
                    total += int(kv) if kv is not None and str(kv).strip() != "" else 0
                except Exception:
                    # als Karakters geen integer is, tel dan 0 mee
                    total += 0
        return total

    def _count_fotos_dr_a_keus(ws):
        class_col = _get_col(ws, "Classificatie")
        beeld_col = _get_col(ws, "Beeld voor print")
        if not class_col or not beeld_col:
            return 0
        target = "A-keus; B-keus; C-keus;"
        dr_values = {"Dragend en bijplaat", "Dragend of bijplaat", "Dragend", "Flexibel"}
        cnt = 0
        for rr in range(2, ws.max_row + 1):
            if ws.cell(rr, class_col).value == target:
                bv = ws.cell(rr, beeld_col).value
                if bv in dr_values:
                    cnt += 1
        return cnt

    def _count_fotos_overig_a_keus(ws):
        class_col = _get_col(ws, "Classificatie")
        beeld_col = _get_col(ws, "Beeld voor print")
        if not class_col or not beeld_col:
            return 0
        target = "A-keus; B-keus; C-keus;"
        cnt = 0
        for rr in range(2, ws.max_row + 1):
            if ws.cell(rr, class_col).value == target:
                bv = ws.cell(rr, beeld_col).value
                if bv is None or str(bv).strip() == "" or bv == "Bijplaat":
                    cnt += 1
        return cnt

    def _count_fotos_flex_a_keus(ws):
        class_col = _get_col(ws, "Classificatie")
        beeld_col = _get_col(ws, "Beeld voor print")
        if not class_col or not beeld_col:
            return 0
        target = "A-keus; B-keus; C-keus;"
        cnt = 0
        for rr in range(2, ws.max_row + 1):
            if ws.cell(rr, class_col).value == target:
                bv = ws.cell(rr, beeld_col).value
                if bv == "Flexibel":
                    cnt += 1
        return cnt



    for ri, tab in enumerate(ordered_tabs, start=2):
        ws_tab = wb[tab]
        ws_stats.cell(row=ri, column=1, value=tab)

        st_aantal = _count_a_keus(ws_tab)
        st_kar = _sum_karakters_a_keus(ws_tab)
        st_fotos_dr = _count_fotos_dr_a_keus(ws_tab)
        st_fotos_overig = _count_fotos_overig_a_keus(ws_tab)
        st_fotos_flex = _count_fotos_flex_a_keus(ws_tab)

        ws_stats.cell(row=ri, column=2, value=st_aantal)
        ws_stats.cell(row=ri, column=3, value=st_kar)
        ws_stats.cell(row=ri, column=4, value=st_fotos_dr)
        ws_stats.cell(row=ri, column=5, value=st_fotos_overig)
        ws_stats.cell(row=ri, column=6, value=st_fotos_flex)

        try:
            comp = 25 - (float(st_kar) / 1000.0)
            if st_fotos_dr == 2:
                comp += 2
            if st_fotos_dr == 1:
                comp += 5
            if (st_fotos_dr + st_fotos_overig) < 3.1:
                comp += 1
            if st_aantal < 3.1:
                comp += 1
            if st_fotos_flex == 2:
                comp -= 1
            if st_fotos_flex > 2.9:
                comp -= 2
        except Exception:
            comp = None
        complexity_map[tab] = comp
        ws_stats.cell(row=ri, column=7, value=comp)
        # v28/29: Akpp_ondergrens, Akpp_range, Akpp_range_boost
        akpp_onder = None
        akpp_range = None
        akpp_range_boost = None
        try:
            if st_kar is not None:
                akpp_onder = (float(st_kar) * 0.7) / 2.0
                if akpp_onder > 4500:
                    akpp_onder = 4500
                akpp_onder = int(round(akpp_onder))
                akpp_range = f"{akpp_onder}:7200"
                # v29: boost = min+300 en max-100
                try:
                    _min_s, _max_s = akpp_range.split(":")
                    _min_v = int(float(_min_s))
                    _max_v = int(float(_max_s))
                    akpp_range_boost = f"{_min_v + 300}:{_max_v - 100}"
                except Exception:
                    akpp_range_boost = None
        except Exception:
            akpp_onder = None
            akpp_range = None
            akpp_range_boost = None
        akpp_range_map[tab] = akpp_range
        akpp_range_boost_map[tab] = akpp_range_boost
        ws_stats.cell(row=ri, column=8, value=akpp_onder)
        ws_stats.cell(row=ri, column=9, value=akpp_range)
        ws_stats.cell(row=ri, column=10, value=akpp_range_boost)
        
        
        
            
    # v28/29: kopieer Akpp_range naar AG1 en Akpp_range_boost naar AH1 op elk tabblad
    for tab in ordered_tabs:
        rng = akpp_range_map.get(tab)
        rng_boost = akpp_range_boost_map.get(tab)
        if tab in wb.sheetnames:
            if rng is not None:
                wb[tab]["AG1"].value = rng
            if rng_boost is not None:
                wb[tab]["AH1"].value = rng_boost
    
    # === Tabblad Planningsvolgorde maken (v23) ===
    plan_name = "Planningsvolgorde"
    if plan_name in wb.sheetnames:
        wb.remove(wb[plan_name])
    ws_plan = wb.create_sheet(title=plan_name)

    ws_plan["A1"] = "Planningsvolgorde"

    # Headerkop opmaak (zoals andere tabbladen): Bold + 20% grijs
    _hdr_font = Font(bold=True)
    _hdr_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    ws_plan["A1"].font = _hdr_font
    ws_plan["A1"].fill = _hdr_fill
    ws_plan["A2"] = "GO-01"
    ws_plan["A5"] = "GO-02"

    other_tabs = ["ZU-MH", "ZU-PS", "ZU-SG", "NM-MI", "NM-NO"]
    other_tabs = [t for t in other_tabs if t in complexity_map]

    other_tabs_sorted = sorted(
        other_tabs,
        key=lambda t: (complexity_map.get(t) is None, -(complexity_map.get(t) or 0)),
    )

    target_rows = [3, 4, 6, 7, 8]
    for r, t in zip(target_rows, other_tabs_sorted):
        ws_plan.cell(row=r, column=1, value=t)

    # === Tabblad Logfile maken (v25) ===
    log_name = "Logfile"
    if log_name in wb.sheetnames:
        wb.remove(wb[log_name])
    ws_log = wb.create_sheet(title=log_name)
    ws_log["A1"] = "Timestamp"
    ws_log["B1"] = "Beschrijving"


    # Headerkop opmaak (zoals andere tabbladen): Bold + 20% grijs
    _hdr_font = Font(bold=True)
    _hdr_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    ws_log["A1"].font = _hdr_font
    ws_log["A1"].fill = _hdr_fill
    ws_log["B1"].font = _hdr_font
    ws_log["B1"].fill = _hdr_fill
    # === Tabbladen rangschikken (v25) ===
    order_names = []
    for fixed in ["Stats", "Logfile", "Planningsvolgorde", "Totale verhalenlijst"]:
        if fixed in wb.sheetnames:
            order_names.append(fixed)

    # Lees de planningvolgorde (A2 t/m A100) en voeg bestaande tabbladen toe in die volgorde
    planned = []
    for r in range(2, 101):
        v = ws_plan.cell(row=r, column=1).value
        if v is None or str(v).strip() == "":
            continue
        name = str(v).strip()
        if name in wb.sheetnames and name not in planned:
            planned.append(name)

    for name in planned:
        if name not in order_names:
            order_names.append(name)

    # Voeg overige tabbladen toe die nog niet genoemd zijn (volgorde zoals ze nu zijn)
    for ws in wb.worksheets:
        if ws.title not in order_names:
            order_names.append(ws.title)

    # Pas de volgorde toe
    wb._sheets = [wb[n] for n in order_names]



    # v27: output-bestandsnaam = Verhalenaanbod_YYYY-MM-DD.xlsx, datum uit Stats!AA1
    final_output_xlsx = output_xlsx
    try:
        out_dir = Path(output_xlsx).parent if output_xlsx else Path(".")
        aa_val = ws_stats["AA1"].value if "ws_stats" in locals() else None
        if isinstance(aa_val, (datetime.date, datetime.datetime)):
            date_str = aa_val.strftime("%Y-%m-%d")
        elif aa_val is not None and str(aa_val).strip():
            date_str = str(aa_val).strip()[-10:].replace(".", "-")
        else:
            date_str = None
        if date_str:
            final_output_xlsx = str((out_dir / f"Verhalenaanbod_{date_str}.xlsx").resolve())
    except Exception:
        final_output_xlsx = output_xlsx

    wb.save(final_output_xlsx)
    return mapped_rows, warnings, final_output_xlsx


In [ ]:
mapped_rows, warnings, OUTPUT_XLSX = process_kordiam(INPUT_XLSX, OUTPUT_XLSX, MAPPING_XLSX)
print(f"Klaar. Gemapte rijen: {mapped_rows}")
if warnings:
    print("\n".join(warnings))
print(f"Output opgeslagen als: {OUTPUT_XLSX}")

# === Download knop ===
try:
    import ipywidgets as widgets
    from IPython.display import display
    out_path = Path(OUTPUT_XLSX)
    if out_path.exists():
        data = out_path.read_bytes()
        dl = widgets.FileDownload(data=data, filename=out_path.name, description="Download output")
        display(dl)
    else:
        print(f"Outputbestand niet gevonden op: {out_path}")
except Exception as e:
    from IPython.display import FileLink, display
    try:
        display(FileLink(OUTPUT_XLSX))
    except Exception:
        print("Kon geen downloadknop of link tonen. Fout:", e)


# Krantenplanner – Constraint-based planning & matching engine (Colab)

Dit notebook reproduceert dezelfde matching-logica als de huidige output (`Krantenplanning_v4.xlsx`).

## Upload-volgorde (exact 5 uploads)
1. Templates (xlsx)
2. Beslispad Spread (xlsx)
3. Beslispad EP (xlsx)
4. Posities en kenmerken (xlsx)
5. Verhalenaanbod / Planningsoverzicht (xlsx)

Elke upload gebeurt in een aparte codecel. Daarna draai je één keer de **RUN**-cel om `Krantenplanning.xlsx` te genereren.

In [ ]:
#@title 1) Upload Templates (xlsx)
from google.colab import files
uploaded = files.upload()

# Verwacht: 1 xlsx bestand
TEMPLATES_PATH = next(iter(uploaded.keys()))
print("Templates:", TEMPLATES_PATH)


In [ ]:
#@title 2) Upload Beslispad Spread (xlsx)
from google.colab import files
uploaded = files.upload()

BESLISPAD_SPREAD_PATH = next(iter(uploaded.keys()))
print("Beslispad Spread:", BESLISPAD_SPREAD_PATH)


In [ ]:
#@title 3) Upload Beslispad EP (xlsx)
from google.colab import files
uploaded = files.upload()

BESLISPAD_EP_PATH = next(iter(uploaded.keys()))
print("Beslispad EP:", BESLISPAD_EP_PATH)


In [ ]:
#@title 4) Upload Posities en kenmerken (xlsx)
from google.colab import files
uploaded = files.upload()

POSITIES_PATH = next(iter(uploaded.keys()))
print("Posities en kenmerken:", POSITIES_PATH)


In [ ]:
#@title 5) Upload Verhalenaanbod / Planningsoverzicht (xlsx)

# Dit bestand is het outputbestand van DEF1 (OUTPUT_XLSX)
VERHALENAANBOD_PATH = OUTPUT_XLSX
print("Verhalenaanbod:", VERHALENAANBOD_PATH)


In [ ]:
#@title Install & imports (eenmalig)
!pip -q install openpyxl pandas

import openpyxl
import pandas as pd
import re
import itertools
import datetime
import os


In [ ]:
def copy_storylist(wb, src_name, dst_name):
    if src_name not in wb.sheetnames:
        return
    if dst_name in wb.sheetnames:
        wb.remove(wb[dst_name])
    wb.copy_worksheet(wb[src_name]).title = dst_name

def overwrite_row_by_name(ws, name, vals):
    # Overschrijf bestaande rij (op Naam productie), behoud kolomstructuur
    headers = [ws.cell(1,c).value for c in range(1, ws.max_column+1)]
    name_col = None
    for i,h in enumerate(headers, start=1):
        if str(h).strip().lower() == "naam productie":
            name_col = i
            break
    if name_col is None:
        return
    for r in range(2, ws.max_row+1):
        if normalize(ws.cell(r, name_col).value) == normalize(name):
            for c,h in enumerate(headers, start=1):
                if h in vals:
                    ws.cell(r,c).value = vals[h]
            return
#@title RUN – Genereer Krantenplanning.xlsx
# Deze cel voert de matching-engine uit en schrijft Krantenplanning.xlsx weg.
# Daarna wordt het bestand automatisch aangeboden voor download.


def _header_index(ws):
    hdr = [ws.cell(1,c).value for c in range(1, ws.max_column+1)]
    norm = [normalize(h) for h in hdr]
    return hdr, {n:i+1 for i,n in enumerate(norm)}

def _get_col(idx_map, *names):
    for nm in names:
        key = normalize(nm)
        if key in idx_map:
            return idx_map[key]
    return None

def normalize(x):
    return "" if x is None else str(x).strip()

def truthy(v):
    if v is None: return False
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return v != 0
    return str(v).strip().lower() in ("true","ja","1","yes","waar")

def cell_set(v):
    # Splits cellen zoals "S_nws_0; S_nws_2;" naar set({"S_nws_0","S_nws_2"})
    if v is None: return set()
    return set([p.strip() for p in str(v).split(";") if p.strip()])

def find_col_any(df, substrings, required=True):
    for sub in substrings:
        for c in df.columns:
            if sub.lower() in str(c).lower():
                return c
    raise KeyError(f"Geen kolom gevonden voor: {substrings}")

def _pretty_class_token(t):
    t = normalize(t)
    if t == "a-keus":
        return "A-keus"
    if t == "b-keus":
        return "B-keus"
    return t


def _split_multi(val):
    if val is None:
        return set()
    s = str(val).strip()
    if s == "":
        return set()
    # accepteer ; , + / en ' en ' als scheiding
    parts = re.split(r"[;,+/]|en", s, flags=re.IGNORECASE)
    return {normalize(p) for p in parts if normalize(p)}

def class_allowed(pc, allowed):
    # allowed kan zijn "Alle" of bv. "A-keus" of meerdere waarden
    if allowed is None or str(allowed).strip()=="":
        return True
    if str(allowed).strip().lower()=="alle":
        return True
    allowed_set = _split_multi(allowed)
    pc_set = _split_multi(pc)
    # match zodra er overlap is
    return len(pc_set & allowed_set) > 0


def parse_range(s):
    if s is None: return None
    s=str(s).strip()
    m=re.match(r"^\s*(\d+(?:\.\d+)?)\s*:\s*(\d+(?:\.\d+)?)\s*$", s)
    if not m: return None
    return (float(m.group(1)), float(m.group(2)))

# -----------------------------
# Load workbooks (data_only=True is essentieel i.v.m. formules zoals Posities!M1)
# -----------------------------
plan_wb = openpyxl.load_workbook(VERHALENAANBOD_PATH, data_only=True)
pos_wb  = openpyxl.load_workbook(POSITIES_PATH, data_only=True)
tmpl_wb = openpyxl.load_workbook(TEMPLATES_PATH, data_only=True)

pos_ws   = pos_wb["Blad1"]
# Helper: always resolve the current Logfile worksheet (important after fallback reset)
def get_log_ws():
    if "Logfile" not in plan_wb.sheetnames:
        plan_wb.create_sheet("Logfile", 0)
        ws = plan_wb["Logfile"]
        ws.append(["Timestamp","Beschrijving"])
    return plan_wb["Logfile"]

# Reset logfile (laat header staan)
_log_ws = get_log_ws()
if _log_ws.max_row > 1:
    _log_ws.delete_rows(2, _log_ws.max_row)

def log(msg):
    ws = get_log_ws()
    row = ws.max_row + 1
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ws.cell(row=row, column=1, value=ts)
    ws.cell(row=row, column=2, value=msg)

# -----------------------------
# Planning order
# -----------------------------
po_ws = plan_wb["Planningsvolgorde"]
order = []
r = 2
while True:
    v = po_ws[f"A{r}"].value
    if v is None or str(v).strip()=="":
        break
    order.append(str(v).strip())
    r += 1

# -----------------------------
# -----------------------------
# Regime berekenen (VERWIJDERD)
# In deze versie is er geen onderscheid meer tussen Normaal/Verhalenschaarste/Papierschaarste.
# Akpp-range wordt per run afgelezen vanaf het run-tabblad (AG1/AH1).
# -----------------------------

# Templates inlezen
# -----------------------------
tmpl_ws = tmpl_wb["Blad1"]
tmpl_headers = [tmpl_ws.cell(1,c).value for c in range(1, tmpl_ws.max_column+1)]
def tcol(name): return tmpl_headers.index(name)+1

templates = []
for rr in range(2, tmpl_ws.max_row+1):
    tpl = tmpl_ws.cell(rr, tcol("Template")).value
    if tpl is None: 
        continue
    placeholders = []
    for i in range(1,6):
        v = tmpl_ws.cell(rr, tcol(f"Placeholder {i}")).value
        if v not in (None,""):
            placeholders.append(str(v).strip())
    templates.append({
        "Template": str(tpl).strip(),
        "Templatesoort": normalize(tmpl_ws.cell(rr, tcol("Templatesoort")).value),
        "Placeholders": placeholders,
        "Advertentiepositie": normalize(tmpl_ws.cell(rr, tcol("Advertentiepositie")).value),
        "Akpp": tmpl_ws.cell(rr, tcol("Akpp")).value,
    })

# -----------------------------
# Beslispaden inlezen (pandas)
# -----------------------------
bps = pd.read_excel(BESLISPAD_SPREAD_PATH, sheet_name="Blad1")
bpe = pd.read_excel(BESLISPAD_EP_PATH, sheet_name="Blad1")

# Zorg dat stapcodes als tekst blijven (BPS1.001 / BPE1.001)
if 'Stappen' in bps.columns:
    bps['Stappen'] = bps['Stappen'].astype(str)
if 'Stappen' in bpe.columns:
    bpe['Stappen'] = bpe['Stappen'].astype(str)

def map_cols(df):
    # Robust tegen aangepaste kolomkoppen (zoals jouw update)
    return {
        "mode": find_col_any(df, ["Bovenste producties"]),
        "skip_pos": find_col_any(df, ["Sla deze stap over bij deze posities"], required=False),
        "class": find_col_any(df, ["Toegestane Classificatie"]),
        "tsoort": find_col_any(df, ["Toegestane 'Templatesoort'", "Templatesoort"]),
        # Spread-kop kan door updates veranderen, daarom zoeken we breed:
        "tw1_if": find_col_any(df, ["Beeld voor print=Bijplaat", "wordt voldaan", "toegestaan if"]),
        "tw1": find_col_any(df, ["Bij maximaal 1 van de placeholders op het template 'Tweede keus placeholder' toegestaan "]),
        "tw2": find_col_any(df, ["Bij maximaal 2 van de placeholders"]),
        "open_xs": find_col_any(df, ["XS_0 open"]),
        "open_s": find_col_any(df, ["S_nws_0 of S_lk_0 open", "S_nws_0"]),
        "open_custom": find_col_any(df, ["Toegestaan om maximaal 1 placeholder op te laten van onderstaande soort(en)"]),
        "derde": find_col_any(df, ["Derde keus placeholder"]),
        "derde_max1": find_col_any(df, ["Bij maximaal 1 van de placeholders op het template \'Derde keus placeholder\' toegestaan"]),
        "vierde": find_col_any(df, ["Bij maximaal 1 van de placeholders op het template 'VIERDE keus placeholder' toegestaan"]),
        "admatch": find_col_any(df, ["Advertentiepositie' matchen", "Advertentiepositie"]),
        "akpp": find_col_any(df, ["Uitsluitend templates toegestaan met Akpp-waarde binnen deze range", "Akpp-waarde binnen deze range", "Akpp range"], required=False),
        "conct": find_col_any(df, ["Concessies_beschreven"]),
    }

cols_bps = map_cols(bps)
cols_bpe = map_cols(bpe)

def resolve_akpp_range(code_or_range, tabname):
    """Los een Akpp-range op voor deze run.

    - Directe range: 'min:max' -> (min,max)
    - Code 'Akpp_range' -> lees van tabblad <tabname> cel AG1
    - Code 'Akpp_range_boost' -> lees van tabblad <tabname> cel AH1
    """
    if code_or_range is None:
        return None
    s = str(code_or_range).strip()
    # 1) direct 'min:max'
    direct = parse_range(s)
    if direct:
        return direct

    code = s.lower().replace(" ", "")
    # 2) via run-tabblad
    try:
        ws_run = plan_wb[tabname]
    except Exception:
        ws_run = None

    if code in ("akpp_range", "akpprange"):
        if ws_run is None:
            return None
        return parse_range(ws_run["AG1"].value)

    if code in ("akpp_range_boost", "akpprange_boost", "akpprangeboost"):
        if ws_run is None:
            return None
        return parse_range(ws_run["AH1"].value)

    # Onbekende code: geen Akpp-filter
    return None

# -----------------------------
# Posities kolommen
# -----------------------------
pos_header = [pos_ws.cell(1,c).value for c in range(1, pos_ws.max_column+1)]
def pos_col(sub):
    for i,h in enumerate(pos_header, start=1):
        if h and sub.lower() in str(h).lower():
            return i
    raise KeyError(sub)

POS_VORM_COL = pos_col("Verschijningsvorm")
POS_POS_COL  = pos_col("Positie")
POS_AD1_COL  = pos_col("Advertentieaanbod")
POS_AD2_COL  = pos_col("tweede keus")
POS_AD3_COL  = pos_col("derde keus")
POS_AD4_COL  = pos_col("vierde keus")

def get_pos_row(posname):
    rr=2
    while True:
        v = pos_ws.cell(rr, POS_POS_COL).value
        if v is None or str(v).strip()=="":
            return None
        if str(v).strip()==posname:
            return rr
        rr += 1

# -----------------------------
# Sheet <-> DataFrame
# -----------------------------
def sheet_to_df(ws):
    headers = [ws.cell(1,c).value for c in range(1, ws.max_column+1)]
    data=[]
    for rr in range(2, ws.max_row+1):
        row = [ws.cell(rr,c).value for c in range(1, ws.max_column+1)]
        if all(v is None or str(v).strip()=="" for v in row):
            continue
        data.append(row)
    return pd.DataFrame(data, columns=headers)

def df_to_sheet(ws, df):
    ws.delete_rows(2, ws.max_row)
    for i,row in enumerate(df.itertuples(index=False), start=2):
        for j,val in enumerate(row, start=1):
            ws.cell(i,j,value=val)

def get_copy_targets(ws):
    # Tabbladen om NAAR TE KOPIËREN (stap 7) – uit cel AA1
    v = ws["AA1"].value
    return [s.strip() for s in re.split(r"[;,]", str(v)) if s.strip()] if v else []

def get_keep_targets(ws):
    # Tabbladen die NIET gestript mogen worden (stap 8) – uit cel AB1
    v = ws["AB1"].value
    return [s.strip() for s in re.split(r"[;,]", str(v)) if s.strip()] if v else []

    rr = ws.max_row + 1
    for c,val in enumerate(vals, start=1):
        ws.cell(rr,c,value=val)

def remove_from_sheet(ws, name):
    # Verwijder rij op basis van kolom 'Naam productie' (ongeacht kolomvolgorde)
    headers = [ws.cell(1,c).value for c in range(1, ws.max_column+1)]
    name_col = None
    for i,h in enumerate(headers, start=1):
        if normalize(h).lower() == "naam productie":
            name_col = i
            break
    if name_col is None:
        name_col = 1  # fallback: oude structuur

    target = normalize(name)
    for rr in range(2, ws.max_row+1):
        if normalize(ws.cell(rr, name_col).value) == target:
            ws.delete_rows(rr,1)
            return

# -----------------------------
# Matching per stap
# -----------------------------
def best_match_for_step(tabname, df_run, step_row, is_spread):
    cols = cols_bps if is_spread else cols_bpe

    mode          = step_row[cols["mode"]]
    is_bovenste   = str(mode).lower().startswith("bovenste")
    allowed_class = step_row[cols["class"]]
    allowed_tsoort= step_row[cols["tsoort"]]
    tw1_if        = truthy(step_row[cols["tw1_if"]])
    tw1           = truthy(step_row[cols["tw1"]])
    tw2           = truthy(step_row[cols["tw2"]])
    open_xs       = truthy(step_row[cols["open_xs"]])
    open_s        = truthy(step_row[cols["open_s"]])
    open_custom_raw = step_row[cols["open_custom"]] if "open_custom" in cols else None
    open_custom_set = cell_set(open_custom_raw)
    derde         = truthy(step_row[cols["derde"]])
    derde_max1    = truthy(step_row[cols["derde_max1"]])
    vierde        = truthy(step_row[cols["vierde"]])
    admatch       = normalize(step_row[cols["admatch"]])
    conc_txt      = step_row[cols["conct"]]

    rng_cell = step_row[cols.get("akpp")] if cols.get("akpp") is not None else None
    rng_pair = resolve_akpp_range(rng_cell, tabname)

    pr = get_pos_row(tabname)
    ad1 = normalize(pos_ws.cell(pr, POS_AD1_COL).value) if pr else ""
    ad2 = normalize(pos_ws.cell(pr, POS_AD2_COL).value) if pr else ""
    ad3 = normalize(pos_ws.cell(pr, POS_AD3_COL).value) if pr else ""
    ad4 = normalize(pos_ws.cell(pr, POS_AD4_COL).value) if pr else ""
    ad_choice = {"Advertentieaanbod":ad1, "Advertentieaanbod_tweede keus":ad2, "Advertentieaanbod_derde keus":ad3, "Advertentieaanbod_vierde keus":ad4}.get(admatch, ad1)

    # IF Advertentieaanbod != W00 THEN range-min -1000
    if rng_pair and ad_choice and ad_choice!="W00":
        rng_pair = (rng_pair[0]-1000, rng_pair[1])

    def template_allowed(t):
        if allowed_tsoort:
            # Sta ";" en "," toe als scheiding in Toegestane Templatesoort
            allowed_tsoort_set = {normalize(p) for p in re.split(r"[;,]", str(allowed_tsoort)) if str(p).strip()}
            if normalize(t["Templatesoort"]) not in allowed_tsoort_set:
                return False
        if ad_choice and normalize(t["Advertentiepositie"]) != ad_choice:
            return False
        if rng_pair:
            try:
                ak = float(t["Akpp"])
            except:
                return False
            return rng_pair[0] <= ak <= rng_pair[1]
        return True

    tpls = [t for t in templates if template_allowed(t)]
    if not tpls:
        return None

    pool = df_run.copy()
    if "Classificatie" in pool.columns:
        pool = pool[pool["Classificatie"].apply(lambda x: class_allowed(x, allowed_class))]
    pool = pool.reset_index(drop=True)
    if pool.empty:
        return None

    max_tw  = 2 if tw2 else (1 if (tw1 or tw1_if) else 0)
    max_der = 0 if not derde else (1 if derde_max1 else len(phs))
    max_v   = 1 if vierde else 0

    # Precompute placeholder sets
    for col in ["Gewenste placeholder","Tweede keus placeholder","Derde keus placeholder","Vierde keus placeholder"]:
        pool[col+"_set"] = pool[col].apply(cell_set) if col in pool.columns else [set()]*len(pool)

    def matches(prow, ph, kind):
        return ph in prow[{"g":"Gewenste placeholder_set","t":"Tweede keus placeholder_set","d":"Derde keus placeholder_set","v":"Vierde keus placeholder_set"}[kind]]

    best = None
    for t in tpls:
        phs = t["Placeholders"]
        k = len(phs)
        if is_bovenste:
            # Neem bovenste N producties (N = aantal placeholders),
            # maar bij ex aequo Prioscore: neem ook alle extra producties met dezelfde Prioscore
            pool_use = pool
            if k < len(pool_use):
                try:
                    boundary = float(pool_use.loc[k-1].get("Prioscore", None))
                    last = k-1
                    while last + 1 < len(pool_use):
                        nxt = pool_use.loc[last+1].get("Prioscore", None)
                        try:
                            nxt_f = float(nxt)
                        except Exception:
                            break
                        if nxt_f == boundary:
                            last += 1
                        else:
                            break
                    pool_use = pool_use.iloc[:last+1]
                except Exception:
                    pool_use = pool_use.iloc[:k]
        else:
            pool_use = pool

        def _bovenste_ok(selected_indices):
            # Regels voor 'Bovenste' met ex aequo:
            # - als je r producties gebruikt, dan mag je NIET een hogere prioscore overslaan.
            # - je mag alleen wisselen binnen de boundary-tie (zelfde Prioscore als de r-de productie).
            try:
                r = len(selected_indices)
                if r == 0:
                    return True
                boundary = float(pool.loc[r-1].get("Prioscore", None))
                # verplicht: alle indices met Prioscore > boundary moeten geselecteerd zijn (prefix)
                mandatory_count = 0
                for j in range(len(pool)):
                    try:
                        pj = float(pool.loc[j].get("Prioscore", None))
                    except Exception:
                        break
                    if pj > boundary:
                        mandatory_count += 1
                    else:
                        break
                for j in range(mandatory_count):
                    if j not in selected_indices:
                        return False
                # toegestaan: indices t/m last waarbij Prioscore == boundary (ties)
                last = r-1
                while last + 1 < len(pool):
                    try:
                        nxt = float(pool.loc[last+1].get("Prioscore", None))
                    except Exception:
                        break
                    if nxt == boundary:
                        last += 1
                    else:
                        break
                return all((i <= last) for i in selected_indices)
            except Exception:
                # fallback: strict prefix-regel zonder ties
                r = len(selected_indices)
                if r == 0:
                    return True
                return set(range(r)).issubset(set(selected_indices))

        # Candidate list per placeholder
        candidates=[]
        for ph in phs:
            c=[]
            for idx, prow in pool_use.iterrows():
                if matches(prow, ph, "g"):
                    c.append((idx,"g")); 
                    continue
                if max_tw>0:
                    if tw1_if:
                        # Let op: dit blijft exact "bijplaat" (zoals in de huidige output-logica)
                        if normalize(prow.get("Beeld voor print","")).lower()=="bijplaat" and matches(prow, ph, "t"):
                            c.append((idx,"t"))
                    else:
                        if matches(prow, ph, "t"):
                            c.append((idx,"t"))
                if max_der>0 and matches(prow, ph, "d"):
                    c.append((idx,"d"))
                if max_v>0 and matches(prow, ph, "v"):
                    c.append((idx,"v"))
            if ph=="XS_0" and open_xs:
                c.append((None,"o"))
            if ph in ("S_nws_0","S_lk_0") and open_s:
                c.append((None,"o"))
            if open_custom_set and ph in open_custom_set:
                c.append((None,"o"))
            candidates.append(c)

        if any(len(c)==0 for c in candidates):
            continue

        # Brute force (max 5 placeholders)
        best_for_tpl=None
        for choice in itertools.product(*candidates):
            idxs=[i for i,_k in choice if i is not None]
            if len(idxs)!=len(set(idxs)): 
                continue  # productie mag niet dubbel
            if sum(1 for _i,k in choice if k=="t")>max_tw: 
                continue
            if sum(1 for _i,k in choice if k=="d")>max_der:
                continue
            if sum(1 for _i,k in choice if k=="v")>max_v:
                continue

            # MAXIMAAL 1× S_nws_0 of S_lk_0 open laten (samen)
            open_s_count = sum(
                1 for slot,(i,k) in enumerate(choice)
                if i is None and k=="o" and phs[slot] in ("S_nws_0","S_lk_0")
            )
            if open_s_count > 1:
                continue

            # MAXIMAAL 1× open laten uit custom lijst (uit beslispad)
            if open_custom_set:
                custom_open_count = sum(
                    1 for slot,(i,k) in enumerate(choice)
                    if i is None and k=="o" and phs[slot] in open_custom_set
                )
                if custom_open_count > 1:
                    continue

            prios=[]
            for i,_k in choice:
                if i is None: 
                    continue
                try:
                    prios.append(float(pool_use.loc[i].get("Prioscore",0)))
                except:
                    prios.append(0.0)
            if not prios:
                continue

            metric=(sum(prios)/len(prios), sum(prios))  # avg, sum
            if best_for_tpl is None or metric>best_for_tpl["metric"]:
                best_for_tpl={"metric":metric, "choice":choice, "pool_use":pool_use}

        if best_for_tpl and (best is None or best_for_tpl["metric"]>best["metric"]):
            best={"metric":best_for_tpl["metric"], "template":t, "choice":best_for_tpl["choice"], "pool_use":best_for_tpl["pool_use"], "conct":conc_txt}

    return best

# -----------------------------
# Execute pipeline
# -----------------------------
success = 0


def _apply_akpp_range_berekening(ws_run, df_run, tabname):
    """
    [AKPP_RANGE_BEREKENING] (versie 16)
    - Karakters_beschikbaar = som(Karakters) voor alle rijen, maar tel NIET mee als 'Telt mee' = nee
    - Restant = [POSITIELIJST] kolom 'Restant' voor dit tabblad
    - Karakters_gewenst_pp = Karakters_beschikbaar / Restant
    - Ondergrens = clip(Karakters_gewenst_pp - 1250, 2700..5200)
    - Bovengrens = clip(Karakters_gewenst_pp + 1250, 5300..7800)
    - Akpp_range = "min:max" -> schrijf naar AG1
    - Akpp_range_boost: min+400 : max-400 -> schrijf naar AH1
    """
    # vind kolommen in df (header-based)
    c_kar = _find_df_col(df_run, "Karakters")
    c_tm  = _find_df_col(df_run, "Telt mee")  # kan ontbreken
    if c_kar is None:
        return

    def _to_float(x):
        try:
            if x is None or (isinstance(x, float) and (x != x)):
                return 0.0
            s = str(x).strip().replace(",", ".")
            return float(s)
        except Exception:
            return 0.0

    kar_sum = 0.0
    for _, row in df_run.iterrows():
        if c_tm is not None:
            tm = normalize(row.get(c_tm, "")).strip().lower()
            if tm == "nee":
                continue
        kar_sum += _to_float(row.get(c_kar, 0))

    # Restant uit POSITIELIJST
    rest_col = None
    for i, h in enumerate(pos_header, start=1):
        if normalize(h).lower() == "restant":
            rest_col = i
            break
    if rest_col is None:
        return
    pr = get_pos_row(tabname)
    if pr is None:
        return
    try:
        restant = int(float(pos_ws.cell(pr, rest_col).value))
    except Exception:
        return
    if restant <= 0:
        return

    gewenst_pp = kar_sum / restant

    onder = max(2700.0, min(5200.0, gewenst_pp - 1250.0))
    boven = max(5300.0, min(7800.0, gewenst_pp + 1250.0))

    onder_i = int(round(onder))
    boven_i = int(round(boven))

    ws_run["AG1"].value = f"{onder_i}:{boven_i}"

    boost_min = onder_i + 400
    boost_max = boven_i - 400
    ws_run["AH1"].value = f"{boost_min}:{boost_max}"


def run_runs(runs):
    global success
    for posname in runs:
        if posname not in plan_wb.sheetnames:
            log(f"Run {posname}: tabblad ontbreekt.")
            continue

        pr = get_pos_row(posname)
        vorm = normalize(pos_ws.cell(pr, POS_VORM_COL).value) if pr else ""

        if vorm.lower()=="niet":
            plan_wb.remove(plan_wb[posname])
            log(f"Run {posname}: Verschijningsvorm=Niet -> verwijderd.")
            continue

        is_spread = (vorm.lower()=="spread")
        beslis = bps if is_spread else bpe
        cols_step = cols_bps if is_spread else cols_bpe

        ws = plan_wb[posname]
        df = sheet_to_df(ws)

        # Speciaal voor NM-U1: bepaal Akpp_range/Akpp_range_boost volgens [AKPP_RANGE_BEREKENING]
        if posname in ("NM-U1","NM-U2","NM-U3","NM-U4","NM-U5","NM-U6","ZU-U1","ZU-U2","ZU-U3","ZU-U4","ZU-U5","ZU-U6"):
            _apply_akpp_range_berekening(ws, df, posname)
        copy_targets = get_copy_targets(ws)
        keep_targets = get_keep_targets(ws)

        matched=False
        for _, step in beslis.iterrows():
            # Optioneel: sla deze stap over voor specifieke posities
            if "skip_pos" in cols_step and cols_step["skip_pos"] is not None:
                raw = step[cols_step["skip_pos"]]
                skip_set = {normalize(s) for s in re.split(r"[;,]", str(raw)) if str(s).strip()} if raw not in (None, "") else set()
                if normalize(posname) in skip_set:
                    continue

            bm = best_match_for_step(posname, df, step, is_spread)
            if bm is None:
                continue

            tpl = bm["template"]
            phs = tpl["Placeholders"]
            choice = bm["choice"]
            pool_use = bm["pool_use"]

            selected=[]
            for slot,(idx,kind) in enumerate(choice):
                if idx is None:
                    continue
                prod = pool_use.loc[idx].to_dict()
                selected.append((normalize(prod.get("Naam productie","")), phs[slot]))

            names_real=[n for n,_ph in selected]

            # Placeholder(s) die open blijven (idx=None)
            open_placeholders=[phs[slot] for slot,(idx,kind) in enumerate(choice) if idx is None]

            df_new = df[df["Naam productie"].astype(str).str.strip().isin(names_real)].copy()
            for col in ["Gekozen template","Gekozen placeholder","Plaatsing"]:
                if col not in df_new.columns:
                    df_new[col]=None

            for i,row in df_new.iterrows():
                nm = normalize(row["Naam productie"])
                df_new.at[i,"Gekozen template"] = tpl["Template"]
                df_new.at[i,"Gekozen placeholder"] = next((ph for n,ph in selected if n==nm), "")
                df_new.at[i,"Plaatsing"] = posname

            # Voeg extra rij(en) toe als er placeholder(s) open blijven
            if open_placeholders:
                # Safety: voorkom dubbele kolomnamen (pandas concat vereist unieke columns)
                if df_new.columns.duplicated().any():
                    df_new = df_new.loc[:, ~df_new.columns.duplicated()].copy()
                for oph in open_placeholders:
                    empty_row={c:None for c in df_new.columns}
                    empty_row["Naam productie"]="NOG ARTIKEL VOOR DEZE PLEK ZOEKEN"
                    empty_row["Gekozen template"]=tpl["Template"]
                    empty_row["Gekozen placeholder"]=oph
                    empty_row["Plaatsing"]=posname
                    df_new = pd.concat([df_new, pd.DataFrame([empty_row])], ignore_index=True)

            df_to_sheet(ws, df_new.reset_index(drop=True))

            
            # Copy / post-processing (standaard vs. UITW-runs)
            name_to_vals={}
            headers = [ws.cell(1,c).value for c in range(1, ws.max_column+1)]
            name_col = None
            for i,h in enumerate(headers, start=1):
                if normalize(h).lower() == "naam productie":
                    name_col = i
                    break
            if name_col is None:
                name_col = 1  # fallback: oude structuur
            for rr in range(2, ws.max_row+1):
                nm = normalize(ws.cell(rr, name_col).value)
                if nm:
                    name_to_vals[nm] = {headers[c-1]: ws.cell(rr,c).value for c in range(1, ws.max_column+1)}

            is_uitw_run = ("-U" in posname)

            if not is_uitw_run:
                # Standaard: kopieer naar targets uit AA1 en overschrijf daar de rij
                for tr in copy_targets:
                    if tr in plan_wb.sheetnames:
                        wst = plan_wb[tr]
                        for nm,_ph in selected:
                            overwrite_row_by_name(wst, nm, name_to_vals[nm])

                # Verwijder gematchte producties van alle andere tabbladen behalve keep_targets (AB1)
                exclude=set([posname]+keep_targets)
                for sh in plan_wb.sheetnames:
                    if sh in exclude:
                        continue
                    wso = plan_wb[sh]
                    for nm in names_real:
                        remove_from_sheet(wso, nm)

            else:
                # UITW-runs: append naar Totale verhalenlijst en strip alleen binnen de UITW-groep (NM-U* of ZU-U*)
                if "Totale verhalenlijst" in plan_wb.sheetnames:
                    wtot = plan_wb["Totale verhalenlijst"]
                    tot_headers = [wtot.cell(1,c).value for c in range(1, wtot.max_column+1)]
                    tot_h2c = {str(h).strip(): i+1 for i,h in enumerate(tot_headers) if h is not None and str(h).strip() != ""}

                    for nm,_ph in selected:
                        vals = name_to_vals.get(nm, {})
                        new_row_idx = wtot.max_row + 1
                        for h, col in tot_h2c.items():
                            if h in vals:
                                wtot.cell(new_row_idx, col).value = vals[h]

                # Strip gematchte producties uit andere tabbladen
                if posname in ("NM-U1","NM-U2","NM-U3","NM-U4","NM-U5","NM-U6","ZU-U1","ZU-U2","ZU-U3","ZU-U4","ZU-U5","ZU-U6"):
                    # volgens spec: strip uit alle andere tabbladen met dezelfde 2 beginletters ("NM")
                    prefix_main = posname[:2] + "-"
                    for sh in plan_wb.sheetnames:
                        if sh == posname or sh == "Totale verhalenlijst":
                            continue
                        if not str(sh).startswith(prefix_main):
                            continue
                        wso = plan_wb[sh]
                        for nm in names_real:
                            remove_from_sheet(wso, nm)
                else:
                    # default (andere -U runs): strip alleen binnen de UITW-groep (NM-U* of ZU-U*)
                    prefix_u = posname[:2] + "-U"  # "NM-U" of "ZU-U"
                    for sh in plan_wb.sheetnames:
                        if sh == posname or sh == "Totale verhalenlijst":
                            continue
                        if not str(sh).startswith(prefix_u):
                            continue
                        wso = plan_wb[sh]
                        for nm in names_real:
                            remove_from_sheet(wso, nm)

            log(f"Run {posname}: stap {step['Stappen']} succesvolle match. Template={tpl['Template']}. {bm['conct']}")
            matched=True
            success += 1
            break

        if not matched:
            log(f"Run {posname}: {beslis.iloc[0]['Stappen']} tot en met {beslis.iloc[-1]['Stappen']} geen match.")



# Volgens Opzet Krantenplanner DL (versimpeld):
# Voer achtereenvolgens 7 runs uit: volgorde uit Planningsvolgorde A2:A8
# (in het planningsbestand: tab 'Planningsvolgorde' -> cellen A2 t/m A8)
runs = order[:7] if len(order) >= 7 else order

run_runs(runs)



print(f"Klaar. Succesvolle matches: {success}")

# -----------------------------
# Save output
# -----------------------------
# -----------------------------
# EXTRA STAP (versie 5):
# Maak kopieën van 'Totale verhalenlijst' -> 'ZU-UITW' en 'NM-UITW'
# en zet Classificatie overal op 'A-keus; B-keus; C-keus;'
# -----------------------------
source_name = "Totale verhalenlijst"
if source_name in plan_wb.sheetnames:
    # verwijder bestaande kopieën
    for nm in ["ZU-UITW", "NM-UITW"]:
        if nm in plan_wb.sheetnames:
            plan_wb.remove(plan_wb[nm])

    src_ws = plan_wb[source_name]

    def _copy_and_set_classificatie(new_title):
        ws_copy = plan_wb.copy_worksheet(src_ws)
        ws_copy.title = new_title

        headers = [ws_copy.cell(1,c).value for c in range(1, ws_copy.max_column+1)]
        class_col = None
        for i,h in enumerate(headers, start=1):
            if ("" if h is None else str(h).strip().lower()) == "classificatie":
                class_col = i
                break
        if class_col is None:
            class_col = ws_copy.max_column + 1
            ws_copy.cell(1, class_col).value = "Classificatie"

        # Nieuwe kolom Z: 'Telt mee' (overal 'ja')
        teltmee_col = 26  # kolom Z
        ws_copy.cell(1, teltmee_col).value = "Telt mee"

        for r in range(2, ws_copy.max_row+1):
            ws_copy.cell(r, class_col).value = "A-keus; B-keus; C-keus;"
            ws_copy.cell(r, teltmee_col).value = "ja"

    _copy_and_set_classificatie("ZU-UITW")
    _copy_and_set_classificatie("NM-UITW")


# -----------------------------
# EXTRA STAP (versie 6):
# Volg [MAPPINGREGELS ZU-UITW]
# -----------------------------
def _find_df_col(df, name):
    tgt = normalize(name).lower()
    for c in df.columns:
        if normalize(c).lower() == tgt:
            return c
    return None

def apply_mappingregels_zu_uitw():
    if "ZU-UITW" not in plan_wb.sheetnames:
        return
    ws = plan_wb["ZU-UITW"]
    df = sheet_to_df(ws)
    if df is None or df.empty:
        return

    # kolommen (robust)
    col_pl = _find_df_col(df, "Plaatsing")
    col_hl = _find_df_col(df, "Heel Limburg")
    col_fr = _find_df_col(df, "Focusregio")
    col_gp = _find_df_col(df, "Gewenste placeholder")
    col_ph_es = _find_df_col(df, "Placeholder bij enigszins geschikt")
    col_pr = _find_df_col(df, "Prioscore")
    col_gt = _find_df_col(df, "Gekozen template")
    col_gph= _find_df_col(df, "Gekozen placeholder")
    col_pd = _find_df_col(df, "Publicatiedwang")
    col_t8 = _find_df_col(df, "Top 8")
    col_tm = _find_df_col(df, "Telt mee")


    # zorg dat kolom "Telt mee" bestaat (default ja)
    if col_tm is None:
        df["Telt mee"] = "ja"
        col_tm = _find_df_col(df, "Telt mee")
    # zorg dat reset-kolommen bestaan
    for cname in ["Prioscore","Gekozen template","Gekozen placeholder","Plaatsing"]:
        if _find_df_col(df, cname) is None:
            df[cname] = None
    col_pr = _find_df_col(df, "Prioscore")
    col_gt = _find_df_col(df, "Gekozen template")
    col_gph= _find_df_col(df, "Gekozen placeholder")
    col_pl = _find_df_col(df, "Plaatsing")

    # 1) Schrap: Plaatsing begint met 'GO-' of 'ZU-'
    if col_pl is not None:
        pl = df[col_pl].fillna("").astype(str)
        df = df[~(pl.str.startswith("GO-") | pl.str.startswith("ZU-"))].copy()

    # 2) Schrap: Heel Limburg=ongeschikt AND Focusregio bevat géén Parkstad/Maastricht/Sittard
    targets = ["Parkstad","Maastricht","Sittard"]
    def _contains_any(txt):
        s = "" if txt is None else str(txt)
        return any(t in s for t in targets)

    if col_hl is not None:
        hl = df[col_hl].fillna("").astype(str).str.strip().str.lower()
        fr = df[col_fr].fillna("").astype(str) if col_fr is not None else pd.Series([""]*len(df), index=df.index)
        mask_bad = (hl == "ongeschikt") & (~fr.apply(_contains_any))
        df = df[~mask_bad].copy()

    # 3) Enigszins geschikt + Focusregio ≠ Sittard/Parkstad/Maastricht -> vervang Gewenste placeholder
    if col_hl is not None and col_fr is not None and col_gp is not None and col_ph_es is not None:
        hl = df[col_hl].fillna("").astype(str).str.strip().str.lower()
        fr = df[col_fr].fillna("").astype(str).str.strip()
        mask = (hl == "enigszins geschikt") & (~fr.isin(targets))
        df.loc[mask, col_gp] = df.loc[mask, col_ph_es]
        if col_tm is not None:
            df.loc[mask, col_tm] = "nee"

    # 4) Wis kolommen Prioscore, Gekozen template, Gekozen placeholder, Plaatsing
    df[col_pr] = None
    df[col_gt] = None
    df[col_gph] = None
    df[col_pl] = None

    # 5) Bereken nieuwe Prioscore
    score = pd.Series([0]*len(df), index=df.index, dtype="int64")

    if col_hl is not None:
        hl = df[col_hl].fillna("").astype(str).str.strip().str.lower()
        score += (hl == "moet mee").astype(int) * 5
        score += (hl == "geschikt").astype(int) * 3
        score += (hl == "enigszins geschikt").astype(int) * 1

    if col_pd is not None:
        pdw = df[col_pd].fillna("").astype(str).str.strip().str.lower()
        score += (pdw == "nee").astype(int) * (-1)

    if col_t8 is not None:
        t8 = df[col_t8].fillna("").astype(str).str.strip().str.lower()
        score += (t8 == "ja").astype(int) * 3

    if col_fr is not None:
        fr = df[col_fr].fillna("").astype(str)
        score += fr.apply(lambda x: 10 if _contains_any(x) else 0).astype(int)

    df[col_pr] = score

    # 6) Sorteer op Prioscore desc
    df = df.sort_values(by=[col_pr], ascending=False, kind="mergesort").reset_index(drop=True)

    df_to_sheet(ws, df)

# run mappingregels
apply_mappingregels_zu_uitw()


# -----------------------------
# EXTRA STAP (versie 7):
# Volg [MAPPINGREGELS NM-UITW]
# -----------------------------
def apply_mappingregels_nm_uitw():
    if "NM-UITW" not in plan_wb.sheetnames:
        return
    ws = plan_wb["NM-UITW"]
    df = sheet_to_df(ws)
    if df is None or df.empty:
        return

    # kolommen (robust)
    col_pl = _find_df_col(df, "Plaatsing")
    col_hl = _find_df_col(df, "Heel Limburg")
    col_fr = _find_df_col(df, "Focusregio")
    col_gp = _find_df_col(df, "Gewenste placeholder")
    col_ph_es = _find_df_col(df, "Placeholder bij enigszins geschikt")
    col_pr = _find_df_col(df, "Prioscore")
    col_gt = _find_df_col(df, "Gekozen template")
    col_gph= _find_df_col(df, "Gekozen placeholder")
    col_pd = _find_df_col(df, "Publicatiedwang")
    col_t8 = _find_df_col(df, "Top 8")
    col_tm = _find_df_col(df, "Telt mee")


    # zorg dat kolom "Telt mee" bestaat (default ja)
    if col_tm is None:
        df["Telt mee"] = "ja"
        col_tm = _find_df_col(df, "Telt mee")
    # zorg dat reset-kolommen bestaan
    for cname in ["Prioscore","Gekozen template","Gekozen placeholder","Plaatsing"]:
        if _find_df_col(df, cname) is None:
            df[cname] = None
    col_pr = _find_df_col(df, "Prioscore")
    col_gt = _find_df_col(df, "Gekozen template")
    col_gph= _find_df_col(df, "Gekozen placeholder")
    col_pl = _find_df_col(df, "Plaatsing")

        # 1) Schrap: Plaatsing begint met 'GO-' of 'NM-'
    if col_pl is not None:
        pl = df[col_pl].fillna("").astype(str)
        df = df[~(pl.str.startswith("GO-") | pl.str.startswith("NM-"))].copy()

    # 2) Schrap: Heel Limburg=ongeschikt AND Focusregio bevat géén Noord/Midden
    targets = ["Noord","Midden"]
    def _contains_any_nm(txt):
        s = "" if txt is None else str(txt)
        return any(t in s for t in targets)

    if col_hl is not None:
        hl = df[col_hl].fillna("").astype(str).str.strip().str.lower()
        fr = df[col_fr].fillna("").astype(str) if col_fr is not None else pd.Series([""]*len(df), index=df.index)
        mask_bad = (hl == "ongeschikt") & (~fr.apply(_contains_any_nm))
        df = df[~mask_bad].copy()

    # 3) Enigszins geschikt + Focusregio ≠ Noord/Midden -> vervang Gewenste placeholder
    if col_hl is not None and col_fr is not None and col_gp is not None and col_ph_es is not None:
        hl = df[col_hl].fillna("").astype(str).str.strip().str.lower()
        fr = df[col_fr].fillna("").astype(str).str.strip()
        mask = (hl == "enigszins geschikt") & (~fr.isin(targets))
        df.loc[mask, col_gp] = df.loc[mask, col_ph_es]
        if col_tm is not None:
            df.loc[mask, col_tm] = "nee"

    # 4) Wis kolommen Prioscore, Gekozen template, Gekozen placeholder, Plaatsing
    df[col_pr] = None
    df[col_gt] = None
    df[col_gph] = None
    df[col_pl] = None

    # 5) Bereken nieuwe Prioscore
    score = pd.Series([0]*len(df), index=df.index, dtype="int64")

    if col_hl is not None:
        hl = df[col_hl].fillna("").astype(str).str.strip().str.lower()
        score += (hl == "moet mee").astype(int) * 5
        score += (hl == "geschikt").astype(int) * 3
        score += (hl == "enigszins geschikt").astype(int) * 1

    if col_pd is not None:
        pdw = df[col_pd].fillna("").astype(str).str.strip().str.lower()
        score += (pdw == "nee").astype(int) * (-1)

    if col_t8 is not None:
        t8 = df[col_t8].fillna("").astype(str).str.strip().str.lower()
        score += (t8 == "ja").astype(int) * 3

    if col_fr is not None:
        fr = df[col_fr].fillna("").astype(str)
        score += fr.apply(lambda x: 10 if _contains_any_nm(x) else 0).astype(int)

    df[col_pr] = score

    # 6) Sorteer op Prioscore desc
    df = df.sort_values(by=[col_pr], ascending=False, kind="mergesort").reset_index(drop=True)

    df_to_sheet(ws, df)

# run mappingregels
apply_mappingregels_nm_uitw()


# -----------------------------
# EXTRA STAP (versie 8):
# Kopieer ZU-UITW -> ZU-U1..ZU-U6 en ZU-UNUSED, verwijder daarna ZU-UITW
# Kopieer NM-UITW -> NM-U1..NM-U6 en NM-UNUSED, verwijder daarna NM-UITW
# -----------------------------
def _ensure_fresh_copy(src_name: str, dest_name: str):
    if dest_name in plan_wb.sheetnames:
        plan_wb.remove(plan_wb[dest_name])
    base_ws = plan_wb[src_name]
    new_ws = plan_wb.copy_worksheet(base_ws)
    new_ws.title = dest_name
    return new_ws

def _copy_uitw_to_u_tabs(prefix: str):
    src = f"{prefix}-UITW"
    if src not in plan_wb.sheetnames:
        return
    # maak kopieën
    for i in range(1, 7):
        _ensure_fresh_copy(src, f"{prefix}-U{i}")
    _ensure_fresh_copy(src, f"{prefix}-UNUSED")
    # verwijder bron
    plan_wb.remove(plan_wb[src])

_copy_uitw_to_u_tabs("ZU")
_copy_uitw_to_u_tabs("NM")


# -----------------------------
# 7) Volg [MAPPINGREGELS EXTRA 1]
# -----------------------------
def _pos_header_map(ws):
    hdr = [ws.cell(1,c).value for c in range(1, ws.max_column+1)]
    return {("" if h is None else str(h).strip().lower()): i+1 for i,h in enumerate(hdr)}

def _get_pos_col(ws, header_name: str):
    m = _pos_header_map(ws)
    return m.get(header_name.strip().lower())

def _apply_extra1_for_tab(tabname: str, bonus: int, append_twede: bool):
    if tabname not in plan_wb.sheetnames:
        return
    ws = plan_wb[tabname]
    # header -> col index
    hmap = _pos_header_map(ws)
    def col(*names):
        for n in names:
            c = hmap.get(n.strip().lower())
            if c:
                return c
        return None

    col_hl   = col("Heel Limburg")
    col_prio = col("Prioscore")
    col_gw   = col("Gewenste placeholder")
    col_tw   = col("Tweede keus placeholder")

    if col_hl is None or col_prio is None:
        return  # kan niets doen zonder deze kolommen

    for r in range(2, ws.max_row+1):
        hl = normalize(ws.cell(r, col_hl).value).strip().lower()
        if hl != "moet mee":
            continue

        # Prioscore +bonus
        cur = ws.cell(r, col_prio).value
        try:
            cur_f = float(cur) if cur not in (None, "") else 0.0
        except Exception:
            cur_f = 0.0
        ws.cell(r, col_prio).value = cur_f + bonus

        # optioneel: Tweede keus placeholder toevoegen aan Gewenste placeholder
        if append_twede and col_gw is not None and col_tw is not None:
            gw = "" if ws.cell(r, col_gw).value is None else str(ws.cell(r, col_gw).value).strip()
            tw = "" if ws.cell(r, col_tw).value is None else str(ws.cell(r, col_tw).value).strip()
            if tw:
                # voeg toe als het nog niet in de string zit (case-sensitive behouden)
                if not gw:
                    ws.cell(r, col_gw).value = tw
                elif tw not in gw:
                    sep = "; " if not gw.rstrip().endswith(";") else " "
                    ws.cell(r, col_gw).value = gw + sep + tw

# bepaal tabbladen op basis van [POSITIELIJST] kolom 'Restant'
rest_col = None
# pos_header en pos_ws bestaan al eerder in het notebook
try:
    rest_col = None
    # gebruik bestaande pos_col helper als die bestaat
    if "pos_col" in globals():
        try:
            rest_col = pos_col("Restant")
        except Exception:
            rest_col = None
    if rest_col is None:
        hdr = [pos_ws.cell(1,c).value for c in range(1, pos_ws.max_column+1)]
        for i,h in enumerate(hdr, start=1):
            if normalize(h).strip().lower() == "restant":
                rest_col = i
                break
except Exception:
    rest_col = None

if rest_col is not None:
    for rr in range(2, pos_ws.max_row+1):
        tab = normalize(pos_ws.cell(rr, POS_POS_COL).value)
        if not tab:
            continue
        val = pos_ws.cell(rr, rest_col).value
        try:
            v = int(float(val)) if val not in (None, "") else None
        except Exception:
            v = None
        if v in (3,4):
            _apply_extra1_for_tab(tab, bonus=20, append_twede=False)
        elif v in (1,2):
            _apply_extra1_for_tab(tab, bonus=30, append_twede=True)



# EXTRA 1 - stap 3 t/m 7: specifieke bewerkingen op ZU-U1, NM-U1, ZU-U2, NM-U2 + opnieuw sorteren
def _remove_tokens(s, tokens):
    if s is None:
        return s
    txt = str(s)
    for tok in tokens:
        # verwijder token met evt. whitespace eromheen, behoud reststring
        txt = re.sub(rf"\b{re.escape(tok)}\s*;\s*", "", txt)
    txt = re.sub(r"\s{2,}", " ", txt).strip()
    return txt

def _apply_extra1_tab(tabname: str, focus_remove_list, focus_bonus_list, rules):
    """
    rules: list van dicts met keys:
      - hl_values: tuple van hl-waarden (lowercase) waarop regel geldt
      - remove_tokens: lijst tokens om te verwijderen uit Classificatie (bij focus NOT contains focus_remove_list)
    """
    if tabname not in plan_wb.sheetnames:
        return
    ws = plan_wb[tabname]
    df = sheet_to_df(ws)
    if df is None or df.empty:
        return

    c_hl  = _find_df_col(df, "Heel Limburg")
    c_fr  = _find_df_col(df, "Focusregio")
    c_cl  = _find_df_col(df, "Classificatie")
    c_pr  = _find_df_col(df, "Prioscore")

    if c_hl is None or c_fr is None or c_pr is None or c_cl is None:
        return

    def contains_any(text, needles):
        if text is None:
            return False
        t = str(text)
        return any(n in t for n in needles)

    for i in range(len(df)):
        hl = "" if df.at[i, c_hl] is None else str(df.at[i, c_hl]).strip().lower()
        fr = "" if df.at[i, c_fr] is None else str(df.at[i, c_fr])

        # verwijder tokens afhankelijk van HL, maar alleen als focusregio géén van de targets bevat
        if not contains_any(fr, focus_remove_list):
            for rule in rules:
                if hl in rule["hl_values"]:
                    df.at[i, c_cl] = _remove_tokens(df.at[i, c_cl], rule["remove_tokens"])

        # +5 bij focusregio match (bonuslijst)
        if contains_any(fr, focus_bonus_list):
            try:
                cur = float(df.at[i, c_pr]) if df.at[i, c_pr] not in (None, "") else 0.0
            except Exception:
                cur = 0.0
            df.at[i, c_pr] = cur + 5

    # sorteer op Prioscore (hoogste bovenaan)
    def prio_key(x):
        try:
            return float(x)
        except Exception:
            return -1e18
    df["_prio_sort"] = df[c_pr].apply(prio_key)
    df = df.sort_values("_prio_sort", ascending=False).drop(columns=["_prio_sort"]).reset_index(drop=True)
    df_to_sheet(ws, df)

# 3) ZU-U1
_apply_extra1_tab(
    "ZU-U1",
    focus_remove_list=["Parkstad","Maastricht","Sittard","Limburg-breed"],
    focus_bonus_list=["Parkstad","Maastricht","Sittard"],
    rules=[
        {"hl_values": ("geschikt",), "remove_tokens": ["A-keus"]},
        {"hl_values": ("enigszins geschikt",), "remove_tokens": ["A-keus","B-keus"]},
    ],
)

# 4) NM-U1
_apply_extra1_tab(
    "NM-U1",
    focus_remove_list=["Noord","Midden","Limburg-breed"],
    focus_bonus_list=["Noord","Midden"],
    rules=[
        {"hl_values": ("geschikt",), "remove_tokens": ["A-keus"]},
        {"hl_values": ("enigszins geschikt",), "remove_tokens": ["A-keus","B-keus"]},
    ],
)

# 5) ZU-U2
_apply_extra1_tab(
    "ZU-U2",
    focus_remove_list=["Parkstad","Maastricht","Sittard","Limburg-breed"],
    focus_bonus_list=["Parkstad","Maastricht","Sittard","Limburg-breed"],
    rules=[
        {"hl_values": ("enigszins geschikt",), "remove_tokens": ["A-keus"]},
    ],
)

# 6) NM-U2
_apply_extra1_tab(
    "NM-U2",
    focus_remove_list=["Noord","Midden","Limburg-breed"],
    focus_bonus_list=["Noord","Midden","Limburg-breed"],
    rules=[
        {"hl_values": ("enigszins geschikt",), "remove_tokens": ["A-keus"]},
    ],
)



# 7) Rangschik alle U-tabbladen op Prioscore (hoogste bovenaan) — geldt ook voor NM-U3 e.v.
def _sort_sheet_by_prioscore(tabname: str):
    if tabname not in plan_wb.sheetnames:
        return
    ws = plan_wb[tabname]
    df = sheet_to_df(ws)
    if df is None or df.empty:
        return
    c_pr = _find_df_col(df, "Prioscore")
    if c_pr is None:
        return

    def prio_key(x):
        try:
            return float(x)
        except Exception:
            return -1e18

    df["_prio_sort"] = df[c_pr].apply(prio_key)
    df = df.sort_values("_prio_sort", ascending=False, kind="mergesort").drop(columns=["_prio_sort"]).reset_index(drop=True)
    df_to_sheet(ws, df)

for sh in list(plan_wb.sheetnames):
    if re.match(r"^(NM|ZU)-U(\d+|UNUSED)$", sh):
        _sort_sheet_by_prioscore(sh)

# 8) Voer de volgende run uit: NM-U1
run_runs(["NM-U1","NM-U2","ZU-U1","ZU-U2"]) 


# 9) Volg [OPSPAARREGELS]
def _ws_find_col(ws, header_name: str):
    tgt = ("" if header_name is None else str(header_name)).strip().lower()
    for c in range(1, ws.max_column+1):
        h = ws.cell(1,c).value
        if ("" if h is None else str(h)).strip().lower() == tgt:
            return c
    return None

def _delete_story_id_rows(ws, story_col, target_story_id):
    # delete bottom-up to avoid skipping
    for r in range(ws.max_row, 1, -1):
        v = ws.cell(r, story_col).value
        if v is None:
            continue
        if str(v).strip() == target_story_id:
            ws.delete_rows(r, 1)

if "Totale verhalenlijst" in plan_wb.sheetnames:
    ws_tot = plan_wb["Totale verhalenlijst"]
    col_pd = _ws_find_col(ws_tot, "Publicatiedwang")
    col_pl = _ws_find_col(ws_tot, "Plaatsing")
    col_sid = _ws_find_col(ws_tot, "Story ID")

    if col_pd is not None and col_sid is not None:
        # tel Story ID voorkomen in Totale verhalenlijst
        sid_counts = {}
        for r in range(2, ws_tot.max_row+1):
            sid = ws_tot.cell(r, col_sid).value
            if sid is None or str(sid).strip() == "":
                continue
            key = str(sid).strip()
            sid_counts[key] = sid_counts.get(key, 0) + 1

        # bepaal welke story ids 'op te sparen' zijn en (nog) geen plaatsing hebben
        target_sids = set()
        for r in range(2, ws_tot.max_row+1):
            pdw = ws_tot.cell(r, col_pd).value
            if normalize(pdw).strip().lower() != "op te sparen":
                continue
            plv = ws_tot.cell(r, col_pl).value if col_pl is not None else None
            if plv is not None and str(plv).strip() != "":
                continue  # Plaatsing gevuld -> niet meenemen
            sid = ws_tot.cell(r, col_sid).value
            if sid is None or str(sid).strip() == "":
                continue
            key = str(sid).strip()
            # alleen actie als story id uniek is op Totale verhalenlijst
            if sid_counts.get(key, 0) == 1:
                target_sids.add(key)

        # schrap op U3..U6 tabbladen de rijen met dezelfde Story ID (alleen als uniek op Totale verhalenlijst)
        target_tabs = ["ZU-U3","ZU-U4","ZU-U5","ZU-U6","NM-U3","NM-U4","NM-U5","NM-U6"]
        for tab in target_tabs:
            if tab not in plan_wb.sheetnames:
                continue
            ws = plan_wb[tab]
            c_sid = _ws_find_col(ws, "Story ID")
            if c_sid is None:
                continue
            for sid in target_sids:
                _delete_story_id_rows(ws, c_sid, sid)



# 10) Voer de volgende runs uit: NM-U3, NM-U4, NM-U5, NM-U6, ZU-U3, ZU-U4, ZU-U5, ZU-U6.
run_runs(["NM-U3","NM-U4","NM-U5","NM-U6","ZU-U3","ZU-U4","ZU-U5","ZU-U6"])



# 11) Verwijder op 'Totale verhalenlijst' alle rijen waar kolom 'Plaatsing' leeg is.
#     Sorteer daarna 'Totale verhalenlijst' op 'Naam productie'.
def _cleanup_totale_verhalenlijst():
    sh = "Totale verhalenlijst"
    if sh not in plan_wb.sheetnames:
        return
    ws = plan_wb[sh]
    df = sheet_to_df(ws)
    if df is None or df.empty:
        return

    c_pl = _find_df_col(df, "Plaatsing")
    c_nm = _find_df_col(df, "Naam productie")
    if c_pl is None:
        return

    pl = df[c_pl].fillna("").astype(str).str.strip()
    df = df[pl != ""].copy()

    if c_nm is not None:
        df = df.sort_values(by=[c_nm], ascending=True, kind="mergesort").reset_index(drop=True)

    df_to_sheet(ws, df)

_cleanup_totale_verhalenlijst()

# 12) Kopieer in alle tabbladen waarvoor een run is uitgevoerd de bijbehorende 'Beschrijving' uit Logfile
#     naar cel AD1 van dat tabblad (dus per tabblad de juiste regel, niet de laatste algemene regel).
def _copy_log_beschrijving_to_tabs():
    if "Logfile" not in plan_wb.sheetnames:
        return
    ws_log = plan_wb["Logfile"]

    # zoek kolom 'Beschrijving' op rij 1
    desc_col = None
    for c in range(1, ws_log.max_column + 1):
        if normalize(ws_log.cell(1, c).value).strip().lower() == "beschrijving":
            desc_col = c
            break
    if desc_col is None:
        return

    # tabs waarvoor runs zijn uitgevoerd: Planningsvolgorde + alle U-tabs + GO-01/GO-02 (als aanwezig)
    tabs = []
    seen = set()
    try:
        for t in order:
            if t and str(t) not in seen:
                tabs.append(str(t)); seen.add(str(t))
    except Exception:
        pass
    for sh in plan_wb.sheetnames:
        shs = str(sh)
        if re.match(r"^(NM|ZU)-U(\d+|UNUSED)$", shs) and shs not in seen:
            tabs.append(shs); seen.add(shs)
    for t in ["GO-01", "GO-02"]:
        if t in plan_wb.sheetnames and t not in seen:
            tabs.append(t); seen.add(t)

    # bouw per tab de laatst voorkomende beschrijving: scan Logfile van onder naar boven
    last_for_tab = {t: None for t in tabs}
    # descriptions beginnen in Logfile met 'Run <TAB>:'
    targets = {t: f"Run {t}:" for t in tabs}

    for r in range(ws_log.max_row, 1, -1):
        v = ws_log.cell(r, desc_col).value
        if v in (None, ""):
            continue
        s = str(v)
        for t, prefix in targets.items():
            if last_for_tab[t] is None and s.startswith(prefix):
                last_for_tab[t] = v
        # early exit als alles gevonden
        if all(last_for_tab[t] is not None for t in tabs):
            break

    # schrijf naar AD1
    for t in tabs:
        if t in plan_wb.sheetnames and last_for_tab.get(t) is not None:
            plan_wb[t]["AD1"].value = last_for_tab[t]

_copy_log_beschrijving_to_tabs()



# -----------------------------
# 13) Voer [EINDBEWERKINGEN] uit
# -----------------------------
def _find_ws_col(ws, header_name: str):
    target = str(header_name).strip().lower()
    for c in range(1, ws.max_column + 1):
        h = ws.cell(1, c).value
        if h is None:
            continue
        if str(h).strip().lower() == target:
            return c
    return None

def _ensure_ws_col(ws, header_name: str):
    col = _find_ws_col(ws, header_name)
    if col is not None:
        return col
    col = ws.max_column + 1
    ws.cell(1, col).value = header_name
    return col

def _get_run_tabs_from_logfile():
    if "Logfile" not in plan_wb.sheetnames:
        return []
    ws = plan_wb["Logfile"]
    # zoek kolom 'Beschrijving'
    desc_col = _find_ws_col(ws, "Beschrijving")
    if desc_col is None:
        return []
    tabs = []
    for r in range(2, ws.max_row + 1):
        txt = ws.cell(r, desc_col).value
        if not txt:
            continue
        s = str(txt)
        # verwacht: "Run <TAB> uitgevoerd, ..."
        m = re.match(r"\s*Run\s+([A-Za-z]{2}-[A-Za-z0-9]+)\s+uitgevoerd", s)
        if not m:
            # fallback: "Run <TAB>: ..."
            m = re.match(r"\s*Run\s+([A-Za-z]{2}-[A-Za-z0-9]+)\s*[:]", s)
        if m:
            tabs.append(m.group(1))
    # uniek, behoud volgorde
    seen=set()
    out=[]
    for t in tabs:
        if t not in seen:
            seen.add(t); out.append(t)
    return out

def _apply_placeholder_concessie(ws):
    col_gp = _find_ws_col(ws, "Gekozen placeholder")
    col_wp = _find_ws_col(ws, "Gewenste placeholder")
    if col_gp is None or col_wp is None:
        return
    col_pc = _ensure_ws_col(ws, "Placeholder-concessie")
    for r in range(2, ws.max_row + 1):
        gp = ws.cell(r, col_gp).value
        wp = ws.cell(r, col_wp).value
        gp_s = "" if gp is None else str(gp).strip()
        wp_s = "" if wp is None else str(wp)
        if gp_s and (gp_s not in wp_s):
            ws.cell(r, col_pc).value = "Ja"
        else:
            # laat leeg (geen waarde gespecificeerd in txt)
            ws.cell(r, col_pc).value = None

def _write_naam_van_positie_to_AE1(tabname: str):
    if tabname not in plan_wb.sheetnames:
        return
    # zoek kolom 'Naam van positie' in POSITIELIJST
    nv_col = None
    for i,h in enumerate(pos_header, start=1):
        if normalize(h).strip().lower() == "naam van positie":
            nv_col = i
            break
    if nv_col is None:
        return
    pr = get_pos_row(tabname)
    if pr is None:
        return
    val = pos_ws.cell(pr, nv_col).value
    if val is None:
        return
    plan_wb[tabname]["AE1"].value = val

# 13.1 Placeholder-concessie op alle run-tabbladen + Totale verhalenlijst
run_tabs = _get_run_tabs_from_logfile()
for t in run_tabs:
    if t in plan_wb.sheetnames:
        _apply_placeholder_concessie(plan_wb[t])
# ook op Totale verhalenlijst
if "Totale verhalenlijst" in plan_wb.sheetnames:
    _apply_placeholder_concessie(plan_wb["Totale verhalenlijst"])

# 13.2 Vul op run-tabbladen cel AE1 met 'Naam van positie' uit POSITIELIJST
for t in run_tabs:
    _write_naam_van_positie_to_AE1(t)




# 13.3 Maak tabblad 'Planning print' als kopie van 'Totale verhalenlijst' en zet vooraan
if "Totale verhalenlijst" in plan_wb.sheetnames:
    # verwijder bestaande Planning print indien aanwezig
    if "Planning print" in plan_wb.sheetnames:
        plan_wb.remove(plan_wb["Planning print"])
    ws_src = plan_wb["Totale verhalenlijst"]
    ws_pp = plan_wb.copy_worksheet(ws_src)
    ws_pp.title = "Planning print"
    # positioneer als eerste (meest links)
    try:
        plan_wb._sheets.remove(ws_pp)
        plan_wb._sheets.insert(0, ws_pp)
    except Exception:
        pass

    # 13.4 Verwijder kolommen op Planning print (op basis van headernaam)
    cols_to_remove = [
        "Note","Voorkeurspositie","Printbeeld","Graphic","Karakters","Artikelsoort",
        "Gewenste placeholder","Tweede keus placeholder","Derde keus placeholder","Vierde keus placeholder",
        "Placeholder bij enigszins geschikt","Prioscore"
    ]
    # bouw header->kolomindex map
    headers = [ws_pp.cell(1,c).value for c in range(1, ws_pp.max_column+1)]
    name_to_col = {}
    for i,h in enumerate(headers, start=1):
        if h is None:
            continue
        name_to_col[str(h).strip().lower()] = i
    # verzamel te verwijderen kolommen die bestaan
    cols_idx = []
    for nm in cols_to_remove:
        ci = name_to_col.get(nm.strip().lower())
        if ci is not None:
            cols_idx.append(ci)
    # verwijder van rechts naar links
    for ci in sorted(set(cols_idx), reverse=True):
        ws_pp.delete_cols(ci, 1)

    # 13.5 Sorteer rijen op basis van kolom 'Plaatsing' volgens vaste volgorde
    df_pp = sheet_to_df(ws_pp)
    if df_pp is not None and not df_pp.empty:
        c_pl = _find_df_col(df_pp, "Plaatsing")
        if c_pl is not None:
            order = [
                "NM-NO","NM-MI","ZU-MH","ZU-SG","ZU-PS",
                # legacy/alternatief: ND vs GO
                "GO-01","GO-02","ND-01","ND-02",
                "NM-U1","NM-U2","NM-U3","NM-U4","NM-U5","NM-U6",
                "ZU-U1","ZU-U2","ZU-U3","ZU-U4","ZU-U5","ZU-U6"
            ]
            rank = {p:i for i,p in enumerate(order)}
            def _rk(x):
                if x is None:
                    return 10**9
                s = str(x).strip()
                return rank.get(s, 10**9)
            df_pp["_pl_rank"] = df_pp[c_pl].apply(_rk)
            df_pp = df_pp.sort_values(by=["_pl_rank"], ascending=True, kind="mergesort").drop(columns=["_pl_rank"]).reset_index(drop=True)
            df_to_sheet(ws_pp, df_pp)



# 13.6 Kopieer 'Totale verhalenlijst' naar 'ZU-VERV' en 'NM-VERV' (en 'GO-VERV' alleen als er een GO-tabblad actief is)
def _copy_sheet(src_name: str, dest_name: str, make_first: bool=False):
    if src_name not in plan_wb.sheetnames:
        return None
    if dest_name in plan_wb.sheetnames:
        plan_wb.remove(plan_wb[dest_name])
    ws_new = plan_wb.copy_worksheet(plan_wb[src_name])
    ws_new.title = dest_name
    if make_first:
        # zet meest links
        plan_wb._sheets.remove(ws_new)
        plan_wb._sheets.insert(0, ws_new)
    return ws_new

_copy_sheet("Totale verhalenlijst", "ZU-VERV")
_copy_sheet("Totale verhalenlijst", "NM-VERV")

has_go_active = any(str(s).startswith("GO") for s in plan_wb.sheetnames)
if has_go_active:
    _copy_sheet("Totale verhalenlijst", "GO-VERV")

def _filter_verv_by_rules(tabname: str, prefix: str):
    if tabname not in plan_wb.sheetnames:
        return
    ws = plan_wb[tabname]
    df = sheet_to_df(ws)
    if df is None or df.empty:
        return

    c_pl = _find_df_col(df, "Plaatsing")
    c_fr = _find_df_col(df, "Focusregio")
    c_pd = _find_df_col(df, "Publicatiedwang")
    c_hl = _find_df_col(df, "Heel Limburg")

    # 13.7: strip rijen waarvan Plaatsing niet begint met prefix
    if c_pl is not None:
        pl = df[c_pl].fillna("").astype(str).str.strip()
        df = df[pl.str.startswith(prefix)].copy()

    # 13.8: extra stripregels
    def _norm(s):
        return normalize(s).strip().lower()

    if tabname == "NM-VERV":
        # Focusregio = Noord/Midden/Limburg-breed AND Publicatiedwang=Ja
        if c_fr is not None and c_pd is not None:
            fr = df[c_fr].fillna("").astype(str).str.strip()
            pdw = df[c_pd].fillna("").astype(str).apply(_norm)
            mask = fr.isin(["Noord","Midden","Limburg-breed"]) & (pdw == "ja")
            df = df[~mask].copy()
        # Heel Limburg = moet mee
        if c_hl is not None:
            hl = df[c_hl].fillna("").astype(str).apply(_norm)
            df = df[hl != "moet mee"].copy()

    elif tabname == "ZU-VERV":
        if c_fr is not None and c_pd is not None:
            fr = df[c_fr].fillna("").astype(str).str.strip()
            pdw = df[c_pd].fillna("").astype(str).apply(_norm)
            mask = fr.isin(["Parkstad","Sittard","Maastricht","Limburg-breed"]) & (pdw == "ja")
            df = df[~mask].copy()
        if c_hl is not None:
            hl = df[c_hl].fillna("").astype(str).apply(_norm)
            df = df[hl != "moet mee"].copy()

    elif tabname == "GO-VERV":
        if c_pd is not None:
            pdw = df[c_pd].fillna("").astype(str).apply(_norm)
            df = df[pdw != "ja"].copy()


    # 13.9: strip rijen waarvoor Top 8 = Ja (op alle *-VERV tabbladen)
    c_t8 = _find_df_col(df, "Top 8")
    if c_t8 is not None:
        t8 = df[c_t8].fillna("").astype(str).apply(_norm)
        df = df[t8 != "ja"].copy()

    df = df.reset_index(drop=True)
    df_to_sheet(ws, df)

_filter_verv_by_rules("NM-VERV", "NM")
_filter_verv_by_rules("ZU-VERV", "ZU")
if has_go_active:
    _filter_verv_by_rules("GO-VERV", "GO")

OUT_PATH = "Krantenplanning.xlsx"
plan_wb.save(OUT_PATH)


In [ ]:
#@title 8) Download outputbestand
from google.colab import files

KRANTENPLANNING_XLSX = 'Krantenplanning.xlsx'

files.download('Krantenplanning.xlsx')


## DEF3 — PDF Generator


Krantenplanning → PDF-handout (strak modern, voorbeeldpagina modern v3).

In [ ]:

#@title 1) Installeer dependencies
!apt-get -qq update
!apt-get -qq install -y libcairo2 libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf2.0-0 libffi-dev shared-mime-info >/dev/null
!pip -q install pandas openpyxl weasyprint pymupdf tqdm
print("✅ Klaar")


In [ ]:
#@title 2) Upload bestanden (progressbar met %)
from google.colab import files

# Dit bestand komt automatisch uit DEF2 (Krantenplanning.xlsx)
print("Planning-Excel wordt automatisch gebruikt: Krantenplanning.xlsx")

print("\nUpload 1/2: mappingbestand (Hoe vaak komt wat voor.xlsx)")
uploaded_mapping = files.upload()

print("\nUpload 2/2: template-jpg zip (Template jpgs.zip)")
uploaded_templates = files.upload()


In [ ]:

#@title 3) Kies bestanden
import os, glob

all_xlsx = sorted(glob.glob("*.xlsx"), key=os.path.getmtime, reverse=True)
all_zip  = sorted(glob.glob("*.zip"),  key=os.path.getmtime, reverse=True)

PLANNING_XLSX = None
DEFAULT_PLANNING = globals().get("KRANTENPLANNING_XLSX", "Krantenplanning.xlsx")
MAPPING_XLSX  = None
TEMPLATE_ZIP  = None

for f in all_xlsx:
  low=f.lower()
  if "hoe vaak" in low or "komt wat voor" in low:
    MAPPING_XLSX = f
  elif "krantenplanning" in low:
    PLANNING_XLSX = f

for f in all_zip:
  if "template" in f.lower() and f.lower().endswith(".zip"):
    TEMPLATE_ZIP = f

if PLANNING_XLSX is None and os.path.exists(DEFAULT_PLANNING):
  PLANNING_XLSX = DEFAULT_PLANNING

print("PLANNING_XLSX =", PLANNING_XLSX)
print("MAPPING_XLSX  =", MAPPING_XLSX)
print("TEMPLATE_ZIP  =", TEMPLATE_ZIP)

assert PLANNING_XLSX and MAPPING_XLSX and TEMPLATE_ZIP, "Stel de 3 variabelen handmatig in."

In [ ]:

#@title 4) Pak templates uit (met progress) + vind TEMPLATE_DIR
import os, zipfile
from tqdm import tqdm

EXTRACT_DIR = "template_jpgs_extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(TEMPLATE_ZIP, "r") as z:
  for m in tqdm(z.infolist(), desc="Extracting templates", unit="file"):
    z.extract(m, EXTRACT_DIR)

best_dir, best_count = None, 0
for root, _, files in os.walk(EXTRACT_DIR):
  cnt = sum(1 for f in files if f.lower().endswith(".jpg"))
  if cnt > best_count:
    best_dir, best_count = root, cnt

assert best_dir and best_count > 0, "Geen jpgs gevonden."
TEMPLATE_DIR = best_dir
print("✅ TEMPLATE_DIR =", TEMPLATE_DIR, "| jpgs:", best_count)


In [ ]:

#@title 5) Genereer PDF (voorbeeldpagina modern v3 + bijlage)
import os, re
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from weasyprint import HTML
from string import Template

def esc(s):
  if s is None or (isinstance(s, float) and pd.isna(s)): return ""
  return (str(s).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"))

def trunc(s, n=30):
  s = "" if s is None or (isinstance(s, float) and pd.isna(s)) else str(s)
  return s if len(s) <= n else s[:n] + "..."

CODE_RE = re.compile(r"[A-Z]\d{3}[A-Z]")
def extract_codes(val):
  if val is None or (isinstance(val, float) and pd.isna(val)): return []
  return list(dict.fromkeys(CODE_RE.findall(str(val))))

def img_path(code):
  p = os.path.join(TEMPLATE_DIR, f"{code}.jpg")
  return p if os.path.exists(p) else None

wb = load_workbook(PLANNING_XLSX, data_only=True)
sheet_names = set(wb.sheetnames)

def get_ae1(sheet):
  if sheet in sheet_names:
    v = wb[sheet]["AE1"].value
    if v is not None and str(v).strip():
      return str(v).strip()
  return sheet

def get_log_ad(sheet):
  if sheet in sheet_names:
    v = wb[sheet]["AD1"].value
    return "" if v is None else str(v)
  return ""

# mapping
m = pd.read_excel(MAPPING_XLSX)
m.columns = [c.strip() for c in m.columns]
for c in ["Artikelsoort","Artikel","Beeld"]:
  if c in m.columns:
    m[c] = m[c].astype(str).str.strip()
map_dict = m.set_index("Artikelsoort")[["Artikel","Beeld"]].to_dict("index")

# planning print
df = pd.read_excel(PLANNING_XLSX, sheet_name="Planning print")
df["Gekozen placeholder"] = df["Gekozen placeholder"].astype(str).str.strip()
df["Artikel"] = df["Gekozen placeholder"].map(lambda x: map_dict.get(x,{}).get("Artikel",""))
df["Beeld"]   = df["Gekozen placeholder"].map(lambda x: map_dict.get(x,{}).get("Beeld",""))

LOG_RULES = [
  ("CM02","<b>Vormgever:</b> Als noodgreep is een stopper-advertentie ingepland van het formaat W32 (104x70 mm). Plaats die eigen uiting of - beter nog - probeer die stopper overbodig te maken door bijvoorbeeld een foto bij een artikel groter te maken."),
  ("CM03","<b>Vormgever:</b> Als noodgreep is een stopper-advertentie ingepland van het formaat W23 (158x94 mm). Plaats die eigen uiting of - beter nog - probeer die stopper overbodig te maken door bijvoorbeeld een foto bij een artikel groter te maken."),
  ("CM04","<b>Vormgever:</b> Als noodgreep is een stopper-advertentie ingepland van het formaat W16 (266x94 mm). Plaats die eigen uiting of - beter nog - probeer die stopper overbodig te maken door bijvoorbeeld een foto bij een artikel groter te maken."),
  ("CI01","<b>Samensteller:</b> Als noodgreep is een artikel S (zonder beeld) open gelaten."),
  ("CH01","<b>Samensteller:</b> Als noodgreep is een artikel XS (zonder beeld) open gelaten."),
  ("CJ01","<b>Samensteller:</b> Als absolute noodgreep is een fors artikel (M over groter) opengelaten. Bekijk wat er de afgelopen dagen is blijven liggen of overleg met de chef hoe deze ruimte kan worden gevuld."),
]

def build_attention_points(plaatsing, g, codes_u):
  points = []
  e_codes = [c for c in codes_u if c.startswith("E")]

  if len(e_codes) == 2:
    points.append("<b>Vormgever:</b> Niet gekozen voor een spreadtemplate, maar voor twee enkele templates. Mogelijk staan beeld en koppen ongelukkig naast elkaar. Doe indien nodig aanpassing.")

  raw_templates = [t for t in g["Gekozen template"].dropna().astype(str).unique() if t.strip()]
  if any("variant" in t.lower() for t in raw_templates):
    points.append("<b>Vormgever en samensteller:</b> Een of meerdere vormen op de template moeten een handmatige reshape krijgen van nieuws naar lichte kop of omgekeerd. Het betreft daarbij per definitie een vorm van maat S of M.")

  log_str = get_log_ad(str(plaatsing))
  for code, text in LOG_RULES:
    if code in log_str:
      points.append(text)

  has_bvp = "Beeld voor print" in g.columns
  for _, r in g.iterrows():
    phs = "" if pd.isna(r.get("Gekozen placeholder")) else str(r.get("Gekozen placeholder"))
    bvp = "" if (not has_bvp) or pd.isna(r.get("Beeld voor print")) else str(r.get("Beeld voor print")).strip()
    tshort = esc(trunc(r.get("Naam productie")))

    if ("B" in phs) and (bvp != "Dragend en bijplaat"):
      points.append(f"<b>Vormgever en samensteller:</b> Bij artikel '{tshort}' is door de chef geen bijplaat gevraagd, maar in de planning kwam het wel goed uit om die toe te kennen. Beoordeel of er inderdaad nog een tweede beeld bij kan. Zo niet, bouw dan de vorm van dit verhaal enigszins om.")

    if (bvp == "Dragend en bijplaat") and ("B" not in phs):
      points.append(f"<b>Vormgever en samensteller:</b> Bij artikel '{tshort}' is door de chef een bijplaat gevraagd naast het dragende beeld, maar deze bijplaat kon bij de planning niet toegekend worden. Beoordeel of dit problematisch is en bouw de vorm van dit verhaal indien nodig enigszins om.")

    if phs.endswith("0") and (bvp not in ["", "Ongeschikt", "Flexibel"]):
      points.append(f"<b>Vormgever en samensteller:</b> Bij artikel '{tshort}' is door de chef ook Beeld gevraagd, maar dit kon bij de planning niet toegekend worden. Grijp alleen in als dit echt problematisch is.")

    if (not phs.endswith("0")) and (bvp == "Ongeschikt"):
      points.append(f"<b>Vormgever en samensteller:</b> Bij artikel '{tshort}' is door de chef aangegeven dat het beeld niet zo geschikt is voor print, maar als absolute noodgreep is er bij de planning toch voor gekozen om bij dit artikel een kleine plaat te gebruiken. In de uitzonderlijke situatie dat het online-beeld bij het verhaal echt niet kan voor print, los je het op met een stopper-advertentie.")

  return points

def preview_html(codes_u):
  codes = codes_u[:2]
  imgs = [(c, img_path(c)) for c in codes if img_path(c)]
  if not imgs:
    return "<div class='preview'><div class='preview-box'><div style='padding:8px;color:#6b7280;'>Geen preview gevonden voor template.</div></div></div>"

  if len(imgs) == 2 and all(c.startswith('E') for c,_ in imgs):
    return f'''
    <div class="preview">
      <div class="preview-box">
        <table class="preview-table">
          <tr>
            <td><img src="{imgs[0][1]}" alt="{imgs[0][0]}"></td>
            <td class="gap"></td>
            <td><img src="{imgs[1][1]}" alt="{imgs[1][0]}"></td>
          </tr>
        </table>
      </div>
    </div>
    '''
  return f'''
  <div class="preview">
    <div class="preview-box">
      <img class="single" src="{imgs[0][1]}" alt="{imgs[0][0]}">
    </div>
  </div>
  '''


def fmt_dlabel(v):
  if v is None or (isinstance(v, float) and pd.isna(v)): 
    return ""
  # openpyxl may return datetime/date or a string
  try:
    import datetime as _dt
    if isinstance(v, (_dt.datetime, _dt.date)):
      return _dt.datetime(v.year, v.month, v.day).strftime("%-d %b %Y")
  except Exception:
    pass
  return str(v).strip()

# Datum voor header komt uit Krantenplanning -> tabblad 'Stats', cel AA1
date_label = ""
if "Stats" in sheet_names:
  date_label = fmt_dlabel(wb["Stats"]["AA1"].value)

# fallback (alleen als Stats/AA1 leeg is)
if not date_label:
  date_label = " "

# Genereer-stempel onderaan: datum + tijd van NU
_now = datetime.now()
gen_date = _now.strftime("%-d %b %Y")
gen_time = _now.strftime("%H:%M")
footer_text = f"Planning gegenereerd met Krantenplanner V1.1, op {gen_date} om {gen_time}"

cards = []
for plaatsing, g in df.groupby("Plaatsing", sort=True):
  codes=[]
  for t in g["Gekozen template"]:
    codes += extract_codes(t)
  codes_u = list(dict.fromkeys(codes))

  e_codes = [c for c in codes_u if c.startswith("E")]
  is_spread = (any(c.startswith("S") for c in codes_u) or len(e_codes) == 2)
  typ = "SPREAD" if is_spread else "ENKELE PAGINA"
  tpl = " / ".join(codes_u[:2]) + (" …" if len(codes_u) > 2 else "") if codes_u else "—"

  title = esc(get_ae1(str(plaatsing))).upper()
  meta  = f"{esc(plaatsing)} • {typ} • TEMPLATE {esc(tpl)} • {len(g)} ARTIKELEN"

  rows = ""
  for _, r in g.iterrows():
    rows += f'''
    <tr>
      <td class="col-small">{esc(r.get('Artikel'))}</td>
      <td class="col-medium">{esc(r.get('Beeld'))}</td>
      <td>{esc(r.get('Naam productie'))}</td>
      <td class="col-author">{esc(r.get('Auteur'))}</td>
      <td class="col-region">{esc(r.get('Focusregio'))}</td>
    </tr>
    '''

  prev = preview_html(codes_u)
  pts  = build_attention_points(plaatsing, g, codes_u)
  att  = ""
  if pts:
    att = "<div class='attention'><div class='attention-title'>AANDACHTSPUNTEN</div><ul>" + "".join([f"<li>{p}</li>" for p in pts]) + "</ul></div>"

  bottom = f"<div class='bottom'>{prev}{att}</div>" if (prev or att) else ""

  cards.append(f'''
  <div class="card">
    <div class="title">{title}</div>
    <div class="meta">{meta}</div>

    <table class="data">
      <thead>
        <tr>
          <th class="col-small">Artikel</th>
          <th class="col-medium">Beeld</th>
          <th>Titel</th>
          <th class="col-author">Auteur</th>
          <th class="col-region">Focusregio</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>

    {bottom}
  </div>
  ''')

cards_html = "<div class='pagebreak'></div>".join(cards)

def placeholder_to_beeld(ph):
  v = "" if ph is None or (isinstance(ph, float) and pd.isna(ph)) else str(ph).strip()
  m = re.search(r"(\d)\s*$", v)
  if not m:
    return ""
  d = int(m.group(1))
  return "Geen" if d == 0 else f"{d} kolom"

def placeholder_to_artikel(ph):
  v = "" if ph is None or (isinstance(ph, float) and pd.isna(ph)) else str(ph).strip()
  if not v:
    return ""
  parts = v.split("_")
  # drop trailing numeric part like "_4"
  if parts and re.fullmatch(r"\d+", parts[-1] or ""):
    parts = parts[:-1]
  if not parts:
    return ""
  base = parts[0]
  if len(parts) > 1:
    toevoeging = "_".join(parts[1:])
    return f"{base} ({toevoeging})"
  return base

def _pick_nonempty(existing, fallback):
  ex = "" if existing is None or (isinstance(existing, float) and pd.isna(existing)) else str(existing).strip()
  return ex if ex else fallback

def verv_section(sheet_name, heading):
  if sheet_name not in sheet_names:
    return ""
  dfv = pd.read_excel(PLANNING_XLSX, sheet_name=sheet_name).fillna("")
  # ensure expected columns exist
  for col in ["Plaatsing","Gekozen placeholder","Artikel","Beeld","Naam productie","Auteur","Focusregio"]:
    if col not in dfv.columns:
      dfv[col] = ""
  # Fill Artikel/Beeld from Gekozen placeholder when blank
  dfv = dfv.copy()
  dfv["Artikel"] = dfv.apply(lambda r: _pick_nonempty(r.get("Artikel",""), placeholder_to_artikel(r.get("Gekozen placeholder",""))), axis=1)
  dfv["Beeld"]   = dfv.apply(lambda r: _pick_nonempty(r.get("Beeld",""),   placeholder_to_beeld(r.get("Gekozen placeholder",""))), axis=1)

  cols = [("Plaatsing","Positie"),("Artikel","Artikel"),("Beeld","Beeld"),("Naam productie","Titel"),("Auteur","Auteur"),("Focusregio","Regio")]
  header = "".join([f"<th>{dst}</th>" for _,dst in cols])
  rows = ""
  for _, r in dfv.iterrows():
    rows += "<tr>" + "".join([f"<td>{esc(r.get(src,''))}</td>" for src,_ in cols]) + "</tr>"
  return f'''
    <div style="margin-top:18px;">
      <div style="font-weight:800;margin:0 0 8px 0;">{heading}</div>
      <table class="data">
        <thead><tr>{header}</tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
  '''

appendix_html = f'''
<div class="pagebreak"></div>
<div class="card">
  <div class="title">BIJLAGE 1: WAT TE VERVANGEN BIJ LAAT NIEUWS?</div>
  <div class="meta">Indien er in de loop van de avond extra verhalen bij komen ten opzichte van de originele planning, dan kunnen de volgende verhalen wijken:</div>
  {verv_section("NM-VERV","In Noord-Midden")}
  {verv_section("ZU-VERV","In Zuid")}
  {verv_section("GO-VERV","Op gehele oplage")}
</div>
'''



def artikel_label(val):
  v = "" if val is None or (isinstance(val, float) and pd.isna(val)) else str(val).strip()
  if "_" in v:
    a, b = v.split("_", 1)
    return f"{a} ({b})"
  return v

def unused_section(sheet_name, heading):
  if sheet_name not in sheet_names:
    return ""
  dfu = pd.read_excel(PLANNING_XLSX, sheet_name=sheet_name).fillna("")
  # normalize expected columns
  for col in ["Artikelsoort","Beeld voor print","Naam productie","Auteur","Focusregio"]:
    if col not in dfu.columns:
      dfu[col] = ""
  dfu = dfu.copy()
  dfu["Artikel"] = dfu["Artikelsoort"].map(artikel_label)
  cols = [("Artikel","Artikel"), ("Beeld voor print","Beeld"), ("Naam productie","Titel"), ("Auteur","Auteur"), ("Focusregio","Focusregio")]
  header = "".join([f"<th>{esc(dst)}</th>" for _,dst in cols])
  rows = ""
  for _, r in dfu.iterrows():
    rows += "<tr>" + "".join([f"<td>{esc(r.get(src,''))}</td>" for src,_ in cols]) + "</tr>"
  return f'''
    <div style="margin-top:18px;">
      <div style="font-weight:800;margin:0 0 8px 0;">{heading}</div>
      <table class="data">
        <thead><tr>{header}</tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
  '''

appendix2_html = f'''
<div class="pagebreak"></div>
<div class="card">
  <div class="title">BIJLAGE 2: DEZE VERHALEN VIELEN OVER DE RAND</div>
  <div class="meta">Deze verhalen waren wel aangemerkt als bruikbaar, maar hebben bij de planning geen plek toegebeeld gekregen. Dat kan zijn omdat ze weinig prioriteit hadden, maar het kan ook zijn dat dit verhaal steeds pech had dat per pagina met dit verhaal steeds de 'puzzel' niet paste. Beoordeel of er op deze lijst onverhoopt verhalen staan die alsnog mee moeten, en grijp indien nodig in.</div>
  {unused_section("NM-UNUSED","In Noord-Midden:")}
  {unused_section("ZU-UNUSED","In Zuid:")}
</div>
'''

tpl = Template(r'''
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
  @page { size: A4 landscape; margin: 18mm 15mm 18mm 15mm;
    @top-left { content: "Krantenplanning"; font-size: 10pt; font-weight: 700; }
    @top-center { content: "De Limburger · $date_label"; font-size: 10pt; font-weight: 600; color: #333; }
    @top-right { content: "Pagina " counter(page) " / " counter(pages); font-size: 10pt; color: #333; }
      @bottom-center { content: "$footer_text"; font-size: 10pt; font-weight: 600; color: #333; }
}
  body { font-family: Arial, sans-serif; color:#111; font-size: 11px; }

  .content { margin-top: 0; }

  .card {
    background:#f4f8fb; border: 1px solid #e5eef6; border-radius: 10px;
    padding: 12px 14px;
  }
  .title { font-size: 20px; font-weight: 800; margin: 0; }
  .meta { margin-top: 6px; color:#4b5563; font-weight: 600; }

  table.data { width:100%; border-collapse: collapse; margin-top: 10px; background:#fff; border-radius: 8px; overflow:hidden; }
  table.data thead th {
    text-align:left; background:#e6f0f7; color:#0f172a;
    padding: 7px 8px; font-size: 11px; border-bottom: 1px solid #cfe0ee;
  }
  table.data tbody td { padding: 7px 8px; border-bottom: 1px solid #eef2f6; vertical-align: top; }
  table.data tbody tr:last-child td { border-bottom: none; }

  .col-small { width: 55px; white-space: nowrap; }
  .col-medium { width: 90px; white-space: nowrap; }
  .col-author { width: 160px; }
  .col-region { width: 95px; white-space: nowrap; }

  .bottom { display:flex; gap: 14px; margin-top: 12px; align-items:flex-start; }

  .preview {
    border:1px solid #d1d5db; border-radius: 6px;
    padding: 6px; background: #fff; width: 310px;
    overflow: hidden; box-sizing: border-box;
  }
  .preview img { display:block; border-radius: 4px; }
  .preview-box { max-height: 78mm; overflow: hidden; }
  .preview-table { width:100%; border-collapse: collapse; table-layout: fixed; }
  .preview-table td { padding:0; vertical-align: top; text-align: center; }
  .preview-table td.gap { width:2mm; }
  .preview-table img { max-height: 78mm; max-width: 100%; width: auto; height: auto; object-fit: contain; display:inline-block; border-radius: 6px; }
  .preview-box img.single { max-height: 78mm; max-width: 100%; width: auto; height: auto; object-fit: contain; display:block; margin: 0 auto; border-radius: 6px; }

  .attention {
    background:#fff; border: 1px solid #e5e7eb; border-radius: 10px;
    padding: 10px 12px; flex: 1;
  }
  .attention-title { font-weight: 800; margin-bottom: 6px; }
  .attention ul { margin: 0; padding-left: 18px; }
  .attention li { margin-bottom: 6px; }

  .pagebreak { page-break-after: always; }
  .pagebreak:last-child { page-break-after: auto; }
</style>
</head>
<body>

  <div class="content">
    $cards_html
    $appendix_html
  </div>
</body>
</html>
''')

html = tpl.substitute(date_label=esc(date_label), footer_text=esc(footer_text), cards_html=cards_html, appendix_html=appendix_html + appendix2_html)

OUT_HTML = "handout_modern_v3.html"
OUT_PDF  = "handout_modern_v3.pdf"
with open(OUT_HTML, "w", encoding="utf-8") as f:
  f.write(html)

HTML(string=html, base_url=os.getcwd()).write_pdf(OUT_PDF)
print("✅ PDF gemaakt:", OUT_PDF)


In [ ]:

#@title 6) Zelf-check: geen download zonder check
import fitz, os

assert os.path.exists(OUT_PDF), "❌ PDF bestaat niet."
doc = fitz.open(OUT_PDF)
page_count = doc.page_count
img_counts = [len(doc.load_page(i).get_images(full=True)) for i in range(min(page_count, 10))]
doc.close()

print("Pagina's:", page_count)
print("Afbeeldingen op eerste pagina's:", img_counts)

if sum(img_counts) == 0:
  raise RuntimeError("❌ Geen afbeeldingen gevonden in de PDF. Controleer TEMPLATE_DIR / templatecodes.")
print("✅ Check ok: er zitten previews in de PDF. Je kunt nu veilig downloaden.")


In [ ]:

#@title 7) Download PDF
from google.colab import files
files.download(OUT_PDF)
